# Epitope mutation analysis - Preprocess data and summarize across genes

#### Define sets of coord cols for using BioFrame

In [1]:
Query_CoordCols = ("Query_Name", "Query_Start", "Query_End")
HmReg_CoordCols = ("Chr", "Start", "End")
HmRegion_CoordCols = HmReg_CoordCols
Epitope_CoordCols = ("Chrom", "Rv_Start", "Rv_End")
RE_CoordCols = ("seqname", "start_0based", "end_1based")
GenomeAnno_CoordCols = ("Chrom", "Start", "End")


# Import Statements (libraries + Functions)

In [2]:
import numpy as np
import pandas as pd
import vcf
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

%matplotlib inline

In [3]:
import ast

In [4]:
import scipy.stats
import scipy.stats as stats
from statsmodels.stats.proportion import proportions_ztest


In [5]:
import screed
import mmh3

In [6]:
from Bio import SeqIO


In [7]:
import io

In [8]:
import json

In [9]:
# https://bioframe.readthedocs.io/en/latest/guide-intervalops.html
import bioframe as bf
#import bioframe.vis

In [10]:
import subprocess

In [11]:
from pycirclize import Circos


In [15]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch


#### Pandas Viewing Settings

In [16]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

### Import custom utils functions

In [18]:
%load_ext autoreload
%autoreload 2
    
from gcutils.general import Rv_dist
#from gcutils.gubbinsfuncs import get_RecombEvents_From_Gubbins_GFF
#from gcutils.eventparalogcomparison import insertSNPs_IntoRef, insert_ParentSNPs_IntoRef
#from gcutils.eventparalogcomparison import extract_EventInfo, Compare_RE_Vs_Paralog_Hashes
#from gcutils.eventparalogcomparison import compare_EventSeq_kmers_To_Paralogs
#from gcutils.eventparalogcomparison import check_snp_overlap_V2 #, check_snp_overlap

from gcutils.eventtoparalogcomparison import genViz_RecombEvent_Vs_Paralogs_V3
from gcutils.eventtoparalogcomparison import process_and_visualize_event


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Defining Functions

### Define Viz Functions

In [19]:

def generate_Event_GraphicFeatures(i_GC_Events_DF):

    L_GFeats = []
    
    for i, row in i_GC_Events_DF.iterrows():
        
        Rv_Start = row["start_0based"]
        Rv_End = row["end_1based"]
        i_Event_ID = row["EventID"]

        Event_Feat = GraphicFeature(start = Rv_Start , end = Rv_End,
                                      #label = i_Event_ID,
                                      color = "purple",
                                      linecolor = "black")
        
        L_GFeats.append(Event_Feat)

    return L_GFeats


def AddEvents_ToGraphicRecord(Graphic_Record, i_GC_Events_DF):

    L_Event_GFeats = generate_Event_GraphicFeatures(i_GC_Events_DF)
    
    Graphic_Record.features = Graphic_Record.features + L_Event_GFeats

    return Graphic_Record


def generate_Epitope_GraphicFeatures(i_Epitopes_DF):

    L_GFeats = []
    
    for i, row in i_Epitopes_DF.iterrows():
        
        Rv_Start = row["Rv_Start"]
        Rv_End = row["Rv_End"]
        i_epitope_seq = row["Epitope_Seq"]
        i_epitope_ID = row["Epitope_ID"]

        Epitope_Feat = GraphicFeature(start = Rv_Start , end = Rv_End ,
                                      #label = i_epitope_ID,
                                      color = "red",
                                      linecolor = "black")
        
        L_GFeats.append(Epitope_Feat)

    return L_GFeats


def generate_AssayedEpitope_GraphicFeatures(i_Epitopes_DF):

    L_GFeats = []
    
    for i, row in i_Epitopes_DF.iterrows():
        
        Rv_Start = row["Rv_Start"]
        Rv_End = row["Rv_End"]
        i_epitope_seq = row["Epitope_Seq"]
        i_epitope_ID = row["Epitope_ID"]
        i_PosEpitope = row["PosEpitope_Any"]
        if i_PosEpitope == True:
    
            Epitope_Feat = GraphicFeature(start = Rv_Start , end = Rv_End ,
                                          #label = i_epitope_ID,
                                          color = "red",
                                          linecolor = "black")
            
        else:
            Epitope_Feat = GraphicFeature(start = Rv_Start , end = Rv_End ,
                                          #label = i_epitope_ID,
                                          color = "blue",
                                          linecolor = "white")

            
        L_GFeats.append(Epitope_Feat)

    return L_GFeats


def AddEpitopes_ToGraphicRecord(Graphic_Record, i_Epitopes_DF):

    L_Epitope_GFeats = generate_Epitope_GraphicFeatures(i_Epitopes_DF)
    
    Graphic_Record.features = Graphic_Record.features + L_Epitope_GFeats

    return Graphic_Record




### Create general function for subsetting Events DF for a "paralog set"

In [20]:
def subset_EventsDF_ForGenes(i_GRE_DF, gene_set):

    bool_array_gene_set = i_GRE_DF["Overlap_Genes"].str.contains('|'.join(gene_set))
    
    gene_set_Subset_DF = i_GRE_DF[bool_array_gene_set]
    
    return gene_set_Subset_DF


### Define functions for annotating peptide/epitopes by GCE w/ AA mutations

# Parse `Mtb151CI` Isolate Metadata

In [21]:
Repo_DataDir = "../../Data"
InputAsmPath_Dir = f"{Repo_DataDir}/231121.InputAsmTSVs.MtbSetV3.151CI"

MtbSetV3_151CI_InputAsmPATHs_TSV = f"{InputAsmPath_Dir}/231121.MtbSetV3.151CI.HybridAndSRAsm.FAPATHs.V1.tsv"
MtbSetV3_151CI_AsmSumm_TSV = f"{InputAsmPath_Dir}/231121.MtbSetV3.151CI.HybridAsm.AsmSummary.V2.tsv"


### Reading in "WGA151CI_AsmSummary_DF"

In [22]:
WGA151CI_AsmSummary_DF = pd.read_csv(MtbSetV3_151CI_AsmSumm_TSV, sep = "\t")

SampleIDs_151CI_SOI = list( WGA151CI_AsmSummary_DF["SampleID"].values )
WGA151CI_SampleIDs = SampleIDs_151CI_SOI
WGA151CI_AsmSummary_DF.shape


(151, 7)

#### Create SampleID Mapping Dicts

In [23]:
WGA151CI_ID_To_PrimLineage_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'PrimaryLineage']].values)
WGA151CI_ID_To_SubLineage_Dict = dict( WGA151CI_AsmSummary_DF[["SampleID", "Lineage"]].values)
WGA151CI_ID_To_Dataset_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'Dataset_Tag']].values)  
ID_To_PrimLineage_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'PrimaryLineage']].values)
ID_To_SubLineage_Dict = dict( WGA151CI_AsmSummary_DF[["SampleID", "Lineage"]].values)
ID_To_Dataset_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'Dataset_Tag']].values)  


# Import/parse processed H37rv genome annotations

In [24]:
RepoRef_Dir = "../../References"

AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir = f"{RepoRef_Dir}/201027_H37rv_AnnotatedGenes_And_IntergenicRegions"
H37Rv_GenomeAnnotations_Genes_TSV = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.tsv"

## H37Rv Gene Annotations TSV
H37Rv_GenomeAnno_Genes_DF = pd.read_csv(H37Rv_GenomeAnnotations_Genes_TSV, sep = "\t")
#H37Rv_GenomeAnno_Genes_DF["Middle"] = (H37Rv_GenomeAnno_Genes_DF["Start"] + H37Rv_GenomeAnno_Genes_DF["End"]) / 2
#H37Rv_GenomeAnno_Genes_DF["Length"]  = H37Rv_GenomeAnno_Genes_DF["End"] - H37Rv_GenomeAnno_Genes_DF["Start"]

H37Rv_GeneInfo_Subset_DF = H37Rv_GenomeAnno_Genes_DF[["H37rv_GeneID", "Symbol", "Feature", "Functional_Category", "Is_Pseudogene", "Product", "PEandPPE_Subfamily", "ExcludedGroup_Category"]]

RvID_To_Symbol_Dict = dict(H37Rv_GeneInfo_Subset_DF[['H37rv_GeneID', 'Symbol']].values)
Symbol_To_RvID_Dict = dict(H37Rv_GenomeAnno_Genes_DF[['Symbol', 'H37rv_GeneID']].values)
Symbol_To_FuncCat_Dict = dict(H37Rv_GenomeAnno_Genes_DF[['Symbol', 'Functional_Category']].values)

ESX_Genes_List_TSV = f"{RepoRef_Dir}/190927_H37rv_ListOf_ESXgenes.tsv"
Esx_Genes_DF = pd.read_csv(ESX_Genes_List_TSV, sep = '\t')

In [25]:
H37Rv_GenomeAnno_Genes_DF.head(1)

,Chrom,Start,End,Strand,H37rv_GeneID,Symbol,Feature,Functional_Category,Is_Pseudogene,Product,PEandPPE_Subfamily,ExcludedGroup_Category
0,NC_000962.3,0,1524,+,Rv0001,dnaA,CDS,information pathways,No,Chromosomal replication initiator protein DnaA,NaN,NotExcluded


## Define relevant H37Rv gene lists for analysis (PE/PPE, Esx, 13E12 gene)

In [26]:
ListOf_Esx_Symbols = list(Esx_Genes_DF["symbol"].values)
ListOf_Esx_RvIDs = list(Esx_Genes_DF["gene_id"].values)

In [27]:
listOf_PEPPE_Symbols = list( H37Rv_GenomeAnno_Genes_DF.query(" Functional_Category == 'PE/PPE' ")["Symbol"].values )
listOf_PEPPE_RvIDs = list( H37Rv_GenomeAnno_Genes_DF.query(" Functional_Category == 'PE/PPE' ")["H37rv_GeneID"].values )

In [28]:
listOf_13E12_Region_RvIDs = ["Rv0094c", "Rv0095c", "Rv0393", "Rv1572c", "Rv1572c", "Rv1128c", "Rv1148c", "Rv1587c", "Rv1588c", "Rv1702c", "Rv1945", "Rv2100", "Rv3466", "Rv3467"]  


# Parse H37Rv Reference sequences (Genome, Genes, Proteins)

## Parse H37Rv genome sequence (DNA)

In [29]:
from Bio import SeqIO


In [30]:
H37rv_Ref_GBK_PATH = "/n/data1/hms/dbmi/farhat/mm774/References/GCF_000195955.2_ASM19595v2_genomic.gbk"
H37Rv_FA = "/n/data1/hms/dbmi/farhat/mm774/References/GCF_000195955.2_ASM19595v2_genomic.fasta"

H37Rv_Seq = SeqIO.read(H37Rv_FA, "fasta").seq
len(H37Rv_Seq)

4411532

## Parse H37Rv Protein (AA) and gene (DNA) sequences

In [31]:
O2_RefDir = "/n/data1/hms/dbmi/farhat/mm774/References"

MycoBrowser_RefFiles_Dir = f"{O2_RefDir}/190619_Mycobrowser_H37rv_ReferenceFiles"

H37Rv_Proteins_MycoBro_FAA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.fasta"
H37Rv_Proteins_MycoBro_FAA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.esxM_Added.fasta"
H37Rv_Proteins_NCBI_FAA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.fasta"
H37RV_Genes_FA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_genes_v3.fasta"



H37Rv_FAA_PATH = f"{O2_RefDir}/GCF_000195955.2_ASM19595v2_proteins.faa"
H37Rv_FAA_PATH = f"{O2_RefDir}/GCF_000195955.2_ASM19595v2_proteins.esxM_Added.faa"
H37Rv_GBK_PATH = f"{O2_RefDir}/GCF_000195955.2_ASM19595v2_genomic.gbk"


In [32]:
!ls -1 $MycoBrowser_RefFiles_Dir

Mycobacterium_tuberculosis_H37Rv_genes_v3.fasta
Mycobacterium_tuberculosis_H37Rv_genome_v3.fasta
Mycobacterium_tuberculosis_H37Rv_genome_v3.fasta.fai
Mycobacterium_tuberculosis_H37Rv_gff_v3.gff
Mycobacterium_tuberculosis_H37Rv_gff_v3.REP13E12_Regions.gff
Mycobacterium_tuberculosis_H37Rv_proteins_v3.fasta
Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.esxM_Added.fasta
Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.fasta
Mycobacterium_tuberculosis_H37Rv_txt_v3_PEPPE_subfamilies.txt
Mycobacterium_tuberculosis_H37Rv_txt_v3.txt.tsv


### Parse MycoBrowser Protein Seq Ref

In [33]:
dictOf_H37Rv_MycoBrow_ProtSeq = {}

for index, record in tqdm(enumerate(SeqIO.parse(H37Rv_Proteins_MycoBro_FAA, "fasta"))):
    ShortID = record.name
    
    dictOf_H37Rv_MycoBrow_ProtSeq[ShortID] = record.seq


4091it [00:00, 142557.20it/s]


### Parse MycoBrowser Gene Seq Ref

In [34]:
dictOf_H37Rv_MycoBrow_Gene_Seq = {}

for index, record in tqdm(enumerate(SeqIO.parse(H37RV_Genes_FA, "fasta"))):

    ShortID = record.name.split("|")[0]
    dictOf_H37Rv_MycoBrow_Gene_Seq[ShortID] = record.seq


4187it [00:00, 22135.78it/s]


In [35]:
list(dictOf_H37Rv_MycoBrow_Gene_Seq.keys())[:2]

['Rv3728', 'Rv3729']

### Parse NCBI Protein Seq Ref

In [36]:
dictOf_H37Rv_ProtSeq = {}
dictOf_H37Rv_ProtRecord = {}

for index, record in tqdm(enumerate(SeqIO.parse(H37Rv_FAA_PATH, "fasta"))):
    Rec_Description = record.description
    dict_Attr = {}
    for i in Rec_Description.split(" "):
        ###Just looking for line with " " character (as key = value)
        if "=" in i:
            key = i.strip().split("=")[0].strip('"').strip('[')
            value = i.strip().split("=")[1].strip('"').strip(']')
            ###Put them in a dictionnary
            dict_Attr[key]=value
    
    ShortID = dict_Attr["locus_tag"]
    dictOf_H37Rv_ProtSeq[ShortID] = record.seq
    dictOf_H37Rv_ProtRecord[ShortID] = record


3907it [00:00, 81338.30it/s]


In [37]:
#dictOf_H37Rv_ProtSeq["Rv1196"]

In [38]:
#dictOf_H37Rv_ProtSeq["Rv1196"] == dictOf_H37Rv_MycoBrow_ProtSeq["Rv1196"]

In [39]:
dictOf_H37Rv_ProtSeq["Rv1792"]

Seq('MASRFMTDPHAMRDMAGRFEVHAQTVEDEARRMWASAQNISGAGWSGMAEATSLDTMT')

In [40]:
dictOf_H37Rv_MycoBrow_ProtSeq["Rv1792"]

Seq('MASRFMTDPHAMRDMAGRFEVHAQTVEDEARRMWASAQNISGAGWSGMAEATSLDTMT')

# Parse in H37Rv Homology-Map Results (k19w19)

### Define all HmMap file paths

In [41]:
Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V9"

H37_Rv_MM2_HomologyMapping_Dir = f"{Main_Project_Dir}/250901.H37Rv.HomologyMapping.k19w19.ProcessedData.V2"

# Define paths to output TSVS

### Homologous regions (MERGED)
RvHmMap_Merged_ParaRegions_TSV  = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.MergedRegions.ParalogousRegions.k19w19.tsv"
RvHmMap_Merged_LocalRepeats_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.MergedRegions.LocalRepeats.k19w19.tsv"

### Homology map (pairwise alignments)
RvHmMap_Aln_All_TSV           = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.All.tsv"
RvHmMap_Aln_PRs_NoOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.NoOverlap.tsv"
RvHmMap_Aln_LRs_WiOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.WiOverlap.tsv"

RvHmMap_Aln_PRs_NoOverlap_Clustered_TSV     = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.NoOverlap.Clustered.tsv"
RvHmMap_Aln_LRs_WiOverlap_Clustered_TSV     = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.OnlyOverlap.Clustered.tsv"

### Variants from the homology map alignments
RvHmMap_Var_TSV      = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.variants.tsv"
RvHmMap_Var_SNPs_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.variants.snps.tsv"


### Parse in HmRegions (`Paralogous_Regions` and `Local_Repeats`)

In [42]:
HmMapRegs_ParaRegs_k19w19_DF = pd.read_csv(RvHmMap_Merged_ParaRegions_TSV,
                                    sep="\t")
#HmMapRegs_ParaRegs_k19w19_DF["Overlap_Genes"] = HmMapRegs_ParaRegs_k19w19_DF["Overlap_Genes"].fillna("_")

HmMapRegs_ParaRegs_k19w19_DF.shape

(200, 13)

In [43]:
HmMapRegs_LocalRepeats_k19w19_DF = pd.read_csv(RvHmMap_Merged_LocalRepeats_TSV, 
                                        sep="\t")

#HmMapRegs_LocalRepeats_k19w19_DF["Overlap_Genes"] = HmMapRegs_LocalRepeats_k19w19_DF["Overlap_Genes"].fillna("_")

HmMapRegs_LocalRepeats_k19w19_DF.shape

(50, 13)

In [44]:
HmMapRegs_All_LRsPRs_K19w19_DF = pd.concat([HmMapRegs_ParaRegs_k19w19_DF,
                                            HmMapRegs_LocalRepeats_k19w19_DF])

HmMapRegs_All_LRsPRs_K19w19_DF.shape

(250, 13)

In [45]:
HmMapRegs_ParaRegs_k19w19_DF.head(2)

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,num_HmAln_All,num_HmAln_PR_BelowSeqID100,num_HmAln_PR_MinSeqID99,HmRegionNum,HmRegionID
0,0,NC_000962.3,80184,80523,80353.5,339,Rv0071,0,1,1,1,0,PR_HmRegion_000
1,1,NC_000962.3,80623,82664,81643.5,2041,"Rv0072,Rv0073",0,1,1,1,1,PR_HmRegion_001


#### Peak at head of each HmMap Regions DFs

In [46]:
HmMapRegs_ParaRegs_k19w19_DF.head()

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,num_HmAln_All,num_HmAln_PR_BelowSeqID100,num_HmAln_PR_MinSeqID99,HmRegionNum,HmRegionID
0,0,NC_000962.3,80184,80523,80353.5,339,Rv0071,0,1,1,1,0,PR_HmRegion_000
1,1,NC_000962.3,80623,82664,81643.5,2041,"Rv0072,Rv0073",0,1,1,1,1,PR_HmRegion_001
2,2,NC_000962.3,103705,105130,104417.5,1425,"Rv0094c,Rv0095c",0,2,2,2,2,PR_HmRegion_002
3,3,NC_000962.3,149571,149808,149689.5,237,PE_PGRS2,0,1,1,1,3,PR_HmRegion_003
4,4,NC_000962.3,177203,177447,177325.0,244,_,0,1,1,1,4,PR_HmRegion_004


In [47]:
HmMapRegs_LocalRepeats_k19w19_DF.head()

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,num_HmAln_All,num_HmAln_PR_BelowSeqID100,num_HmAln_PR_MinSeqID99,HmRegionNum,HmRegionID
0,0,NC_000962.3,333811,335879,334845.0,2068,PE_PGRS3,0,1,1,1,0,LR_HmRegion_000
1,1,NC_000962.3,366430,375121,370775.5,8691,"PPE5,PPE6",0,6,6,6,1,LR_HmRegion_001
2,2,NC_000962.3,424011,432951,428481.0,8940,"hspR,PPE7,PPE8",0,5,4,4,2,LR_HmRegion_002
3,3,NC_000962.3,566288,580814,573551.0,14526,"hbhA,Rv0476,Rv0477,deoC,Rv0479c,Rv0480c,Rv0481...",0,4,2,1,3,LR_HmRegion_003
4,4,NC_000962.3,631298,631436,631367.0,138,Rv0538,0,2,2,2,4,LR_HmRegion_004


### Parse in homology-map DFs (pairwise alignments between all homologous regions)

In [48]:
HmMap_Aln_k19w19_DF = pd.read_csv(RvHmMap_Aln_All_TSV,
                           sep="\t")
HmMap_Aln_k19w19_DF.shape

(776, 24)

In [49]:
HmMap_Aln_k19w19_NoOverlap_DF = pd.read_csv(RvHmMap_Aln_PRs_NoOverlap_Clustered_TSV,
                                     sep="\t")
HmMap_Aln_k19w19_NoOverlap_DF.shape

(640, 34)

In [50]:
HmMap_Aln_k19w19_LocalRepeat_DF = pd.read_csv(RvHmMap_Aln_LRs_WiOverlap_Clustered_TSV,
                                              sep="\t")
HmMap_Aln_k19w19_LocalRepeat_DF.shape

(136, 34)

In [51]:
HmMap_Aln_PR_ExactCopy_DF = HmMap_Aln_k19w19_NoOverlap_DF.query("SeqID == 1.0")
print(HmMap_Aln_PR_ExactCopy_DF.shape)

(255, 34)


In [52]:
HmMap_Aln_PR_NoPerfAln_DF = HmMap_Aln_k19w19_NoOverlap_DF.query("SeqID != 1.0")
print(HmMap_Aln_PR_NoPerfAln_DF.shape)

(385, 34)


In [53]:

HmMap_Aln_PR_MaxSeqID99_DF = HmMap_Aln_k19w19_NoOverlap_DF.query("SeqID <= 0.99")
print(HmMap_Aln_PR_MaxSeqID99_DF.shape)

(331, 34)


### Parse in HomologyMap Alignment Variants DFs

In [54]:
Mtb_HM_Var_DF = pd.read_csv(RvHmMap_Var_TSV, sep="\t")
Mtb_HM_Var_SNPs_DF = pd.read_csv(RvHmMap_Var_SNPs_TSV, sep="\t")

In [55]:
# Build trimmed + unique HM SNPs DF
UnqSNPs_TarCol = ['Query_Name', 'Query_Start', 'Query_End', 'Ref', 'Alt', 'SNP']

HM_Var_SNPs_TrimUnq_DF = Mtb_HM_Var_SNPs_DF[UnqSNPs_TarCol].drop_duplicates()
HM_Var_SNPs_TrimUnq_DF.shape

(51590, 6)

In [56]:
Mtb_HM_Var_DF.shape

(79508, 13)

In [57]:
Mtb_HM_Var_SNPs_DF.shape

(65617, 13)

# Parse processed epitope info (Panda-24, Lindestam-16)

In [58]:
Repo_Epitope_MainDir = "../../Data/220813_MtbEpitopes"

Lind16_Peptides_HLAInfo_TSV = f"{Repo_Epitope_MainDir}/240815.Lindestram2016.HLA_ResponseInfo.V1.tsv" 

LPM_AllAssayedPeptides_Mapped_TSV = f"{Repo_Epitope_MainDir}/240820.Panda24_Lind16.Merged.PeptidesMappedToRv.AllAssayed.V1.tsv" 

LPM_AllPeptides_Mapped_DF = pd.read_csv(LPM_AllAssayedPeptides_Mapped_TSV, sep = "\t")



In [59]:
LPM_AllPeptides_Mapped_DF.head(4)

,Epitope_ID,Epitope_Seq,Epitope_Len,RvID,Symbol,AA_Start,AA_End,Chrom,Rv_Start,Rv_End,EpitopeSeqFreqInAntigen,Dataset,Assayed_Panda24,PosEpitope_Panda24,Assayed_Lindestam16,PosEpitope_Lindestam16,PosEpitope_Any,EpitopeSymbol_ID,N_HmRegion,HasHmRegion,Antigen_LVL2
0,UnqPeptide_1,FPTLNYAVSVAEACE,15,Rv0322,udgA,368,383,NC_000962.3,390363,390408,1,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_1-udgA,0,False,False
1,UnqPeptide_2,ERIPKFAHLPTVLGE,15,Rv2992c,gltS,238,253,NC_000962.3,3349518,3349563,1,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_2-gltS,0,False,False
2,UnqPeptide_3,FPGVLVAARPVGMFR,15,Rv3628,ppa,66,81,NC_000962.3,4067620,4067665,1,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_3-ppa,0,False,False
3,UnqPeptide_4,GDPARTMRRMIGGLR,15,Rv3617,ephA,167,182,NC_000962.3,4058233,4058278,1,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_4-ephA,0,False,False


In [60]:
LPM_AllPeptides_Mapped_DF.query("Antigen_LVL2 == True")["Symbol"].nunique()

53

In [61]:
LPM_AllPeptides_Mapped_DF.query("PosEpitope_Any == True")["Symbol"].nunique()

135

### Subset epitope mapping data into POSITIVE and NEGATIVEs

In [62]:
LPM_AllPeptides_Mapped_DF.shape

(18741, 21)

In [63]:
LPM_PosEpitopes_Mapped_DF = LPM_AllPeptides_Mapped_DF.query("PosEpitope_Any == True")
LPM_PosEpitopes_Mapped_DF.shape

(424, 21)

In [64]:
LPM_NegPeptides_Mapped_DF = LPM_AllPeptides_Mapped_DF.query("PosEpitope_Any == False")
LPM_NegPeptides_Mapped_DF.shape

(18317, 21)

### Create a Dict that maps each peptide sequence to all gene's that contain the sequence

In [65]:
# Group by 'Epitope_Seq' and aggregate the unique 'Symbol' values into a string
EpiSeq_to_Symbols_dict = (
    LPM_AllPeptides_Mapped_DF.groupby('Epitope_Seq')['Symbol']
    .apply(lambda x: ', '.join(x.unique()))
    .to_dict()
)

In [66]:
EpiSeq_to_Symbols_dict["DLYSKIESLPASQRD"]

'icd2'

## Parse gene-level epitope mapping summary for all H37Rv genes

In [67]:
Repo_Epitope_MainDir = "../../Data/220813_MtbEpitopes"

Rv_Genes_Epitope_SummStats_TSV = f"{Repo_Epitope_MainDir}/240820.RvGene.EpitopeMappingStats.V1.tsv"

Rv_Genes_EpitopeSummary_DF = pd.read_csv(Rv_Genes_Epitope_SummStats_TSV, sep="\t")
Rv_Genes_EpitopeSummary_DF.shape

(3841, 20)

In [68]:
Rv_Genes_EpitopeSummary_DF.head(1)

,Chrom,Start,End,Strand,Feature,H37rv_GeneID,Symbol,Functional_Category,Gene_Cat_V2,N_Pos,N_Neg,Total,Positive_Proportion,Middle,Length,AnyPosEpitope,Antigen_LVL2,N_HmRegion,HasHmRegion,AntigenLVL2_And_HHR_Comb
0,NC_000962.3,0,1524,+,CDS,Rv0001,dnaA,information pathways,information pathways,0,8,8,0.0,762.0,1524,False,False,0,False,NonReactive-Unq


In [69]:
Rv_Genes_EpitopeSummary_DF["Antigen_LVL2"].value_counts()

Antigen_LVL2
False    3788
True       53
Name: count, dtype: int64

In [70]:
Rv_Genes_EpitopeSummary_DF["AntigenLVL2_And_HHR_Comb"].value_counts()

AntigenLVL2_And_HHR_Comb
NonReactive-Unq    3584
NonReactive-HHR     204
Antigen-Unq          33
Antigen-HHR          20
Name: count, dtype: int64

# Define list of HIGH CONFIDENCE antigens based on the requirement of having **2 or more** positive epitopes

In [71]:
Rv_Genes_EpitopeSummary_DF.query("Antigen_LVL2 == True").shape

(53, 20)

In [72]:
Antigens_LVL2 = list(Rv_Genes_EpitopeSummary_DF.query("N_Pos >= 2")["Symbol"].unique())
len(Antigens_LVL2)

53

In [73]:
# #Antigens_LVL2 = list(Rv_Genes_EpitopeSummary_DF.query("Antigen_LVL2 == True")["Symbol"].unique())
# len(Antigens_LVL2)

## parse and process HLA and epitope positivity info for 63 ZA individuals (Supplemental Data from Lindestam-2016)

In [74]:
Lind16_IEDB_AllPeptides_HLAInfo_DF = pd.read_csv(Lind16_Peptides_HLAInfo_TSV, sep = "\t")

Lind16_IEDB_AllPeptides_HLAInfo_DF["Gene(s)"] = Lind16_IEDB_AllPeptides_HLAInfo_DF["Epitope_Seq"].map(EpiSeq_to_Symbols_dict)

Lind16_IEDB_PosEpitopes_HLAInfo_DF = Lind16_IEDB_AllPeptides_HLAInfo_DF.query("Epitope_Status == 'Positive' ")

# Group by "Epitope_Seq" and sum the relevant numeric columns
Lind16_EpiPosFreq_DF = Lind16_IEDB_AllPeptides_HLAInfo_DF.groupby("Epitope_Seq")["Num_Positive"].sum()
Lind16_EpiPosFreq_DF = Lind16_EpiPosFreq_DF.reset_index()

Lind16_EpiPosFreq_DF["Num_Assayed"] = 63

# Assuming Num_Assayed is 63 for each epitope, calculate the fraction of individuals positive for each epitope
Lind16_EpiPosFreq_DF['Fraction_Positive'] = Lind16_EpiPosFreq_DF['Num_Positive'] / 63

Lind16_EpiPosFreq_DF["Gene(s)"] = Lind16_EpiPosFreq_DF["Epitope_Seq"].map(EpiSeq_to_Symbols_dict)

Lind16_EpiPosFreq_DF["Gene(s)"] = Lind16_EpiPosFreq_DF["Gene(s)"].fillna("")


In [75]:
Lind16_IEDB_AllPeptides_HLAInfo_DF.head(3)

,Epitope_Seq,Epitope_Status,HLA_Allele,Num_Assayed,Num_Positive,Fraction_PositiveAssay,Gene(s)
0,DAHGAMIRAQAGSLE,Positive,HLA-DQB1*06:02,5,5,1.0,"esxI, esxV"
1,ISTNIRQAGVQYSRA,Positive,HLA-DQB1*06:02,4,4,1.0,esxB
2,MHVSFVMAYPEMLAA,Positive,HLA-DQB1*06:02,4,4,1.0,NaN


# Parse `Mtb151-MainAnalysis` Gubbins Results

## Define dictionary of file paths for Gubbins analysis

In [76]:
AnalysisName = "250901.WGA151CI.V9"

Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V9"

Target_Output_Dir = f"{Main_Project_Dir}/{AnalysisName}"

WGA151_Gubbins_V1_OutputDir = f"{Target_Output_Dir}/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh"

Gubbins_V1_OutputDir = WGA151_Gubbins_V1_OutputDir

WGA151_Gubbins_OutPrefix = "Gubbins"

WGA151_Gubbins_FullPrefix_PATH  = f"{WGA151_Gubbins_V1_OutputDir}/{WGA151_Gubbins_OutPrefix}"

WGA151_Gubbins_FilePath_Dict = {}

WGA151_Gubbins_FilePath_Dict["NodeLabelledTree_PATH"]           = f"{WGA151_Gubbins_FullPrefix_PATH}.node_labelled.final_tree.tre"
WGA151_Gubbins_FilePath_Dict["NodeToPriLineage_Dict_JSON"]      = f"{WGA151_Gubbins_FullPrefix_PATH}.NodeToPrimaryLineage.json"
WGA151_Gubbins_FilePath_Dict["BranchStats_CSV"]                 = f"{WGA151_Gubbins_FullPrefix_PATH}.per_branch_statistics.csv"
WGA151_Gubbins_FilePath_Dict["BranchStats_WithLineage_CSV"]     = f"{WGA151_Gubbins_FullPrefix_PATH}.per_branch_statistics.WithLineagePerNode.tsv"
WGA151_Gubbins_FilePath_Dict["EventsPer_1kb_H37Rv_TSV"]         = f"{WGA151_Gubbins_FullPrefix_PATH}.H37Rv.EventsPer1kb.tsv"
WGA151_Gubbins_FilePath_Dict["EventsPer_Gene_H37Rv_TSV"]        = f"{WGA151_Gubbins_FullPrefix_PATH}.H37Rv.EventsPerGene.tsv"
WGA151_Gubbins_FilePath_Dict["EventsPer_HHR_H37Rv_TSV"]         = f"{WGA151_Gubbins_FullPrefix_PATH}.H37Rv.EventsPerMergedHomologousRegion.tsv"

WGA151_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"]             = f"{WGA151_Gubbins_FullPrefix_PATH}.recombination_predictions.Anno.tsv"
WGA151_Gubbins_FilePath_Dict["BaseAncRec_All_TSV"]              = f"{WGA151_Gubbins_FullPrefix_PATH}.branch_base_reconstruction.AnnoByEvent.All.tsv"  
WGA151_Gubbins_FilePath_Dict["BaseAncRec_EventsOnly_TSV"]       = f"{WGA151_Gubbins_FullPrefix_PATH}.branch_base_reconstruction.AnnoByEvent.EventSNPsOnly.tsv" 

WGA151_Gubbins_FilePath_Dict["ASR_SNPs_Anno_V2_All_TSV"]        = f"{WGA151_Gubbins_FullPrefix_PATH}.SNPs.AnnoByEvent.AnnoByCDSEffect.All.tsv"
WGA151_Gubbins_FilePath_Dict["ASR_SNPs_Anno_V2_EventsOnly_TSV"] = f"{WGA151_Gubbins_FullPrefix_PATH}.SNPs.AnnoByEvent.AnnoByCDSEffect.EventSNPsOnly.tsv"

RecombPreds_Anno_ByHmMatch_ByEpiMut_V3_TSV = f"{Gubbins_V1_OutputDir}/Gubbins.All_GCEs.AnnoBy.EpitopeEffect.V3.tsv"

WGA151_Gubbins_FilePath_Dict["GCE_WiParalogMap_WiEpitopeOverlap_TSV"] = RecombPreds_Anno_ByHmMatch_ByEpiMut_V3_TSV

######### Add Event to Paralog Mapping Results File Paths ##########

EventMapping_ResultsDir = f"{Target_Output_Dir}/RecombEvent-To-HmRegion-Comparison-V3"

GRE_Anno_ByTopHomologMatch_TSV              = f"{EventMapping_ResultsDir}/GubbinsEvents.WiParalogMapping.V1.tsv"
EventMappingToAllParalogs_Info_TSV_PATH     = f"{EventMapping_ResultsDir}/EventMapping.EventsToAllHmRegions.SeqComparisonInfo.tsv"
Pickle_PATH_dictOf_EventAndHomolog_KmerComp = f"{EventMapping_ResultsDir}/EventMapping.DictOf.KmerComparisons.pickle"   


WGA151_Gubbins_FilePath_Dict["GCEvents_WiParalogMapInfo_TSV"]       = GRE_Anno_ByTopHomologMatch_TSV
WGA151_Gubbins_FilePath_Dict["EventMappingToAllParalogs_TSV"]       = EventMappingToAllParalogs_Info_TSV_PATH
WGA151_Gubbins_FilePath_Dict["EventKmerAnalysis_Dict_PicklePath"]   = Pickle_PATH_dictOf_EventAndHomolog_KmerComp

HmRegions_MappedEvents_TSV = f"{EventMapping_ResultsDir}/GCE.Stats.PerMergedHmRegion.PRs.tsv"
HmPairs_MappedEvents_TSV   = f"{EventMapping_ResultsDir}/GCE.Stats.PerPairwiseAln.PRs.tsv"

WGA151_Gubbins_FilePath_Dict["MergedHmRegion_PRs_GCE_Stats_TSV"] = HmRegions_MappedEvents_TSV
WGA151_Gubbins_FilePath_Dict["HmMapAln_PRs_GCE_Stats_TSV"]       = HmPairs_MappedEvents_TSV

####################################################################


## A) Parse table of ALL Gubbins Events (all putative recomb events) - `WGA-151-GCE` 

In [77]:
import ast

In [78]:
print( WGA151_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"]) 

/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V9/250901.WGA151CI.V9/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh/Gubbins.recombination_predictions.Anno.tsv


In [79]:
# Parse annotated events TSV
GRE_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"], sep = "\t")

# Convert the string column to a list of strings
GRE_DF['taxa_List'] = GRE_DF['taxa_List'].apply(ast.literal_eval)

GRE_DF["LenOfTaxaList"] = GRE_DF["taxa_List"].apply(len)

GRE_DF.shape

(324, 31)

### B) Parse Gubbins recombination events counted over regions (`1 kb window`, `Gene-level`, `HmRegion`)

In [80]:
GRE_GeneLevel_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["EventsPer_Gene_H37Rv_TSV"], sep = "\t")

GRE_GeneLevel_Atleast1_DF = GRE_GeneLevel_DF.query("pGCE_Count > 0")
GRE_GeneLevel_Atleast1_DF.shape

(76, 14)

In [81]:
GRE_PerHHRStats_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["EventsPer_HHR_H37Rv_TSV"] , sep = "\t")
GRE_PerHHRStats_DF.shape

(200, 16)

In [82]:
GRE_PerHHRStats_DF.query("pGCE_Count > 0").shape

(54, 16)

In [83]:
GRE_PerHHRStats_DF["pGCE_Count"].sum()

296

In [84]:
GRE_PerHHRStats_DF.query("pGCE_Count > 0").shape

(54, 16)

In [85]:
GRE_PerHHRStats_DF.head(1)

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,num_HmAln_All,num_HmAln_PR_BelowSeqID100,num_HmAln_PR_MinSeqID99,HmRegionNum,HmRegionID,pGCE_Count,CenterOfRegion,Overlap_GC_EventIDs
0,0,NC_000962.3,80184,80523,80353.5,339,Rv0071,0,1,1,1,0,PR_HmRegion_000,0,80353.5,_


In [86]:
GRE_PerHHRStats_DF["Length"].sum()

256568

In [87]:
257094 / 4411532

0.05827771395515209

In [88]:
256568 / 4411532

0.058158480999344446

## C) Parse Gubbins Events - branch-level stats

In [89]:
G_BranchStats_Filt_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["BranchStats_WithLineage_CSV"], sep = "\t")
G_BranchStats_Filt_DF.shape

(300, 15)

In [90]:
G_BranchStats_Filt_DF.head(1)

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
0,N0072,328,36,292,4,391,3619,0.123288,0.013699,4411487,4408943,lineage1,295.04892,0.0,328


## D) Read in the "node_To_PrimaryLin_Dict" dictionary 

In [91]:
with open(WGA151_Gubbins_FilePath_Dict["NodeToPriLineage_Dict_JSON"]) as json_file:
    node_To_PrimaryLin_Dict = json.load(json_file)

## E) Parse `WGA151` - Gubbins ASR SNP DFs

In [92]:
GubSNPs_All_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["BaseAncRec_All_TSV"], sep = "\t")
GubSNPs_All_DF.shape

(26508, 9)

In [93]:
GubSNPs_EventOnly_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["BaseAncRec_EventsOnly_TSV"], sep = "\t")
GubSNPs_EventOnly_DF.shape

(2916, 9)

In [94]:
GubSNPs_CDSAnno_All_V2_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["ASR_SNPs_Anno_V2_All_TSV"], sep = "\t")
GubSNPs_CDSAnno_All_V2_DF.shape

(26508, 23)

In [95]:
GubSNPs_CDSAnno_EventOnly_V2_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["ASR_SNPs_Anno_V2_EventsOnly_TSV"], sep = "\t")
GubSNPs_CDSAnno_EventOnly_V2_DF.shape

(2916, 23)

## F) Parse `Event-to-Paralog-Mapping` results

### F.1) Event table anno by paralog match

In [96]:
GRE_AnnoByMatch_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["GCEvents_WiParalogMapInfo_TSV"],
                                 sep = "\t")

#GRE_AnnoByMatch_DF["Overlap_Genes"] = GRE_AnnoByMatch_DF["Overlap_Genes"].fillna("None")
#GRE_AnnoByMatch_DF["MaxJC_ToAnyRvGene"] = GRE_AnnoByMatch_DF["MaxJC_ToAnyRvGene"].fillna(0)

GRE_ABM_DF = GRE_AnnoByMatch_DF
GRE_ABM_DF.shape

(324, 46)

### F.2) Event to all paralogs DF

In [97]:
GRE_MatchToHm_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["EventMappingToAllParalogs_TSV"],
                               sep = "\t")

GRE_MatchToHm_DF.shape

(590, 15)

### F.3) Event to paralog details dict

In [98]:
with open(WGA151_Gubbins_FilePath_Dict["EventKmerAnalysis_Dict_PicklePath"], "rb") as f:
    
    dictOf_FullKmerAnalysis_PerEvent = pickle.load(f)
    
print(len(list(dictOf_FullKmerAnalysis_PerEvent.keys()) ))

295


## F.4) HmMap and GCE Stats annotated onto HmRegions and HmMap alignment pairs  
- Parse in mapping results quantified across `Paralogous Regions and each `HmMap-Alignment`

In [99]:
HmPair_EventCt_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["HmMapAln_PRs_GCE_Stats_TSV"],
                                sep = "\t")

HmPair_EventCt_NoPerf_DF = HmPair_EventCt_DF.query("SeqID < 100")
HmPair_EventsOnly_DF = HmPair_EventCt_NoPerf_DF.query("EventsMapped > 0")


HmRegions_DonorAndEventCt_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["MergedHmRegion_PRs_GCE_Stats_TSV"],
                                           sep = "\t")


In [100]:
HmRegions_DonorAndEventCt_DF.head(1)

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,num_HmAln_All,num_HmAln_PR_BelowSeqID100,num_HmAln_PR_MinSeqID99,HmRegionNum,HmRegionID,Norm_Donor_Count,N_Events_Mapped,N_Events_Putative,EventSkew,DonAccCt,EventSkew_Norm,GCE_FractionMapped,Num_PR_SNPs_All,Num_PR_SNPs_Unq,PR_SetID,RPSID
0,0,NC_000962.3,80184,80523,80353.5,339,Rv0071,0,1,1,1,0,PR_HmRegion_000,0.0,0,0,0.0,0.0,NaN,NaN,20,20,PR_Set_1,_


#### Filter All putative GC events to MAPPED GC events

In [101]:
# Seperate GC Event Info DF into GCEs WITH and WITHOUT paralogs
GRE_ABM_WiHm_DF = GRE_ABM_DF.query("NumHmTargets > 0")
GRE_ABM_NoHm_DF = GRE_ABM_DF.query("NumHmTargets == 0")

#
G_Min05KmerMatch_DF = GRE_ABM_WiHm_DF.query("(Max_KmerMatch_ToHm > 0.5)")
G_Filt_DF = G_Min05KmerMatch_DF 
mGCE_DF = G_Filt_DF 

Kmatch_RelevantCols = ["EventID", "EventLen", "snp_count", "seqname", "start_0based", "end_1based", "Parent_Node", "Child_Node",
                       "Overlap_Genes", "Overlap_Gene_RvIDs", "NumHmTargets", "NumHm_AtMaxKmerMatch",
                       "Top_KmerMatch_HomologTarIDs", "Max_KmerMatch_ToHm", 'TotalKmers_Eval']

G_Filt_Kmatch_DF = G_Filt_DF[Kmatch_RelevantCols]
G_Filt_Kmatch_DF.shape[0]

213

In [102]:
pGCE_DF = GRE_ABM_DF 
pGCE_DF.shape

(324, 46)

In [103]:
mGCE_EventIDs = mGCE_DF["EventID"].unique()
len(mGCE_EventIDs)

213

In [104]:
pGCE_EventIDs = pGCE_DF["EventID"].unique()
len(pGCE_EventIDs)

324

### Add antigen-overlap column to RE table

In [105]:
AntigenLVL2_Symbol_pattern = '|'.join(Antigens_LVL2)

mGCE_DF["Overlap_WiAntigenLVL2Gene"]    = mGCE_DF["Overlap_Genes"].str.contains(AntigenLVL2_Symbol_pattern)  

#mGCE_V3_DF["Overlap_WiAntigenLVL2Gene"] = mGCE_V3_DF["Overlap_Genes"].str.contains(AntigenLVL2_Symbol_pattern)  

GRE_ABM_DF["Overlap_WiAntigenLVL2Gene"] = GRE_ABM_DF["Overlap_Genes"].str.contains(AntigenLVL2_Symbol_pattern)  

pGCE_DF["Overlap_WiAntigenLVL2Gene"]    = pGCE_DF["Overlap_Genes"].str.contains(AntigenLVL2_Symbol_pattern)  


/tmp/ipykernel_2183696/2882474862.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mGCE_DF["Overlap_WiAntigenLVL2Gene"]    = mGCE_DF["Overlap_Genes"].str.contains(AntigenLVL2_Symbol_pattern)


In [106]:
pGCE_DF.head(1)

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs,N_LCR_Ovrlap,OvrlapWi_LowComplexityRegion,N_LowPmap_Ovrlap,OvrlapWi_LowPmap,LenOfTaxaList,IsTermNode,Freq_SNP_FoundInAnyPR,NumHmTargets,NumHm_AtMaxKmerMatch,Top_KmerMatch_HomologTarIDs,Top_KmerMatch_HomologGeneIDs,Max_KmerMatch_ToHm,DistToHm_TopKmatch,TotalKmers_Eval,NumHm_AtMaxSNPMatch,Top_SNPMatch_HomologTarIDs,Top_SNPMatch_HomologGeneIDs,Max_SNPMatch_ToHm,DistToHm_TopSNPmatch,MappedEvent,Overlap_WiAntigenLVL2Gene
0,NC_000962.3,103600,104478,0.0,.,Node_148,Node_133,1737.432787,10,"['mada_2-31', 'mada_1-41', 'MT_0080', 'mada_10...",103599,104038.5,879,NaN,"Rv0093c,Rv0094c","Rv0093c,Rv0094c",False,False,True,False,Event_001,2,1,0,0,PR_HmRegion_002,0,0,1,1,130,False,0.0,2,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.8608,NaN,79,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.0,NaN,True,False


In [107]:
mGCE_DF.head(1)

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs,N_LCR_Ovrlap,OvrlapWi_LowComplexityRegion,N_LowPmap_Ovrlap,OvrlapWi_LowPmap,LenOfTaxaList,IsTermNode,Freq_SNP_FoundInAnyPR,NumHmTargets,NumHm_AtMaxKmerMatch,Top_KmerMatch_HomologTarIDs,Top_KmerMatch_HomologGeneIDs,Max_KmerMatch_ToHm,DistToHm_TopKmatch,TotalKmers_Eval,NumHm_AtMaxSNPMatch,Top_SNPMatch_HomologTarIDs,Top_SNPMatch_HomologGeneIDs,Max_SNPMatch_ToHm,DistToHm_TopSNPmatch,MappedEvent,Overlap_WiAntigenLVL2Gene
0,NC_000962.3,103600,104478,0.0,.,Node_148,Node_133,1737.432787,10,"['mada_2-31', 'mada_1-41', 'MT_0080', 'mada_10...",103599,104038.5,879,NaN,"Rv0093c,Rv0094c","Rv0093c,Rv0094c",False,False,True,False,Event_001,2,1,0,0,PR_HmRegion_002,0,0,1,1,130,False,0.0,2,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.8608,NaN,79,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.0,NaN,True,False


# Process Gubbins ASR SNPs DFs and classify by GC Event Group
- A) All SNP mutational events (N = 26,508)
- B) SNP events that cause an AA change (non-synonymous) (N = 13883)
- C) SNP events that cause an AA change in putative GC events (N = 1379)
- D) SNP events that cause an AA change in mapped GC events (N = 994)

In [108]:
Gub_SNPs_NSMutOnly_DF = GubSNPs_CDSAnno_All_V2_DF[ ~GubSNPs_CDSAnno_All_V2_DF["Mut_AA"].isna() ].query("Ref_AA != Mut_AA")
Gub_SNPs_NSMutOnly_DF["EventID"] = Gub_SNPs_NSMutOnly_DF["EventID"].fillna("None")

Gub_SNPs_NSMutIn_pGCE_DF = Gub_SNPs_NSMutOnly_DF.query("EventID != 'None'")
Gub_SNPs_NSMutIn_mGCE_DF = Gub_SNPs_NSMutOnly_DF[ Gub_SNPs_NSMutOnly_DF["EventID"].isin(mGCE_EventIDs) ]


In [109]:
GubSNPs_CDSAnno_All_V2_DF.shape

(26508, 23)

In [110]:
Gub_SNPs_NSMutOnly_DF.shape  

(13883, 23)

In [111]:
Gub_SNPs_NSMutIn_pGCE_DF.shape

(1379, 23)

In [112]:
Gub_SNPs_NSMutIn_mGCE_DF.shape

(994, 23)

In [113]:
LPM_PosEpitopes_Mapped_DF.head(1)

,Epitope_ID,Epitope_Seq,Epitope_Len,RvID,Symbol,AA_Start,AA_End,Chrom,Rv_Start,Rv_End,EpitopeSeqFreqInAntigen,Dataset,Assayed_Panda24,PosEpitope_Panda24,Assayed_Lindestam16,PosEpitope_Lindestam16,PosEpitope_Any,EpitopeSymbol_ID,N_HmRegion,HasHmRegion,Antigen_LVL2
29,UnqPeptide_29,AQAAVVRFQEAANKQ,15,Rv3874,esxB,50,65,NC_000962.3,4352423,4352468,1,Lindestam16_Panda24_Merged,True,False,True,True,True,UnqPeptide_29-esxB,0,False,True


# Part 1: Annotate each epitope by the # of AA mutations & associated GC events

In [114]:
Query_CoordCols = ("Query_Name", "Query_Start", "Query_End")
HmReg_CoordCols = ("Chr", "Start", "End")
HmRegion_CoordCols = HmReg_CoordCols
Epitope_CoordCols = ("Chrom", "Rv_Start", "Rv_End")
RE_CoordCols = ("seqname", "start_0based", "end_1based")
GenomeAnno_CoordCols = ("Chrom", "Start", "End")

Gubbins_SNP_CoordCols = ("Chrom", "Start", "End")


## Analyze overlap of all GC events ACROSS ALL ASSAYED PEPTIDES

In [115]:

LPM_AllPep_V2_DF = bf.count_overlaps(LPM_AllPeptides_Mapped_DF,
                                     Gub_SNPs_NSMutOnly_DF,
                                     cols1 = Epitope_CoordCols,
                                     cols2 = Gubbins_SNP_CoordCols).rename(columns={'count': 'N_NSMut_Total'})


LPM_AllPep_V2_DF = bf.count_overlaps(LPM_AllPep_V2_DF,
                                     Gub_SNPs_NSMutIn_mGCE_DF,
                                     cols1 = Epitope_CoordCols,
                                     cols2 = Gubbins_SNP_CoordCols).rename(columns={'count': 'N_NSMut_mGCE'})

LPM_AllPep_V2_DF["IsMutBymGCE"] =  LPM_AllPep_V2_DF["N_NSMut_mGCE"] > 0

LPM_AllPep_V2_DF = bf.count_overlaps(LPM_AllPep_V2_DF,
                                     Gub_SNPs_NSMutIn_pGCE_DF,
                                     cols1 = Epitope_CoordCols,
                                     cols2 = Gubbins_SNP_CoordCols).rename(columns={'count': 'N_NSMut_pGCE'})

LPM_AllPep_V2_DF["IsMutBypGCE"] =  LPM_AllPep_V2_DF["N_NSMut_pGCE"] > 0


EpiID_To_AAMut_mGCE_IDs_V2 = {}

ListOfRows_2 = []

for j, row_2 in tqdm(LPM_AllPep_V2_DF.iterrows()):
    i_Epi_ID = row_2["Epitope_ID"]
    i_EpiSym_ID = row_2["EpitopeSymbol_ID"]
    
    i_Symbol = row_2["Symbol"]

    i_Rv_Start_Epi = row_2["Rv_Start"]
    i_Rv_End_Epi = row_2["Rv_End"]

    # Define Epitope range
    Epitope_Range = f"NC_000962.3:{i_Rv_Start_Epi}-{i_Rv_End_Epi}"

    # Step 1: 
    
    NS_Mut_mGCE_OnEpitope_DF = bf.select(Gub_SNPs_NSMutIn_mGCE_DF, Epitope_Range, cols=["Chrom", "Start", "End"])

    Ovrlap_mGCE_IDs = list(NS_Mut_mGCE_OnEpitope_DF["EventID"].unique())
    N_Unq_mGCEs = NS_Mut_mGCE_OnEpitope_DF["EventID"].nunique()

    row_2["N_mGCEs_WiNS"] = N_Unq_mGCEs
    row_2["WiNS_mGC_EventIDs"] = ",".join(Ovrlap_mGCE_IDs)

    
    NS_Mut_pGCE_OnEpitope_DF = bf.select(Gub_SNPs_NSMutIn_pGCE_DF, Epitope_Range, cols=["Chrom", "Start", "End"])

    Ovrlap_pGCE_IDs = list(NS_Mut_pGCE_OnEpitope_DF["EventID"].unique())
    N_Unq_pGCEs = NS_Mut_pGCE_OnEpitope_DF["EventID"].nunique()

    row_2["N_pGCEs_WiNS"] = N_Unq_pGCEs
    row_2["WiNS_pGC_EventIDs"] = ",".join(Ovrlap_pGCE_IDs)
    
    EpiID_To_AAMut_mGCE_IDs_V2[i_EpiSym_ID] = Ovrlap_mGCE_IDs
    
    ListOfRows_2.append(row_2)

LPM_AllPep_V2_DF = pd.DataFrame(ListOfRows_2)

LPM_AllPep_V2_DF["WiNS_mGC_EventIDs"] = LPM_AllPep_V2_DF["WiNS_mGC_EventIDs"].replace("", ".").fillna(".")
LPM_AllPep_V2_DF["WiNS_pGC_EventIDs"] = LPM_AllPep_V2_DF["WiNS_pGC_EventIDs"].replace("", ".").fillna(".")

LPM_AllPep_V2_DF.shape

18741it [00:44, 424.43it/s]


(18741, 30)

### Look at `LPM_AllPep_V2_DF`

In [116]:
#LPM_AllPep_V2_DF["IsMutByGCE"] =  LPM_AllPep_V2_DF["N_NSMut_mGCE"] > 0


In [117]:
LPM_AllPep_V2_DF["N_mGCEs_WiNS"].value_counts().head(4)

N_mGCEs_WiNS
0    18615
1       77
2       21
3       14
Name: count, dtype: int64

In [118]:
LPM_AllPep_V2_DF.head(4)

,Epitope_ID,Epitope_Seq,Epitope_Len,RvID,Symbol,AA_Start,AA_End,Chrom,Rv_Start,Rv_End,EpitopeSeqFreqInAntigen,Dataset,Assayed_Panda24,PosEpitope_Panda24,Assayed_Lindestam16,PosEpitope_Lindestam16,PosEpitope_Any,EpitopeSymbol_ID,N_HmRegion,HasHmRegion,Antigen_LVL2,N_NSMut_Total,N_NSMut_mGCE,IsMutBymGCE,N_NSMut_pGCE,IsMutBypGCE,N_mGCEs_WiNS,WiNS_mGC_EventIDs,N_pGCEs_WiNS,WiNS_pGC_EventIDs
0,UnqPeptide_1,FPTLNYAVSVAEACE,15,Rv0322,udgA,368,383,NC_000962.3,390363,390408,1,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_1-udgA,0,False,False,0,0,False,0,False,0,.,0,.
1,UnqPeptide_2,ERIPKFAHLPTVLGE,15,Rv2992c,gltS,238,253,NC_000962.3,3349518,3349563,1,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_2-gltS,0,False,False,2,0,False,0,False,0,.,0,.
2,UnqPeptide_3,FPGVLVAARPVGMFR,15,Rv3628,ppa,66,81,NC_000962.3,4067620,4067665,1,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_3-ppa,0,False,False,0,0,False,0,False,0,.,0,.
3,UnqPeptide_4,GDPARTMRRMIGGLR,15,Rv3617,ephA,167,182,NC_000962.3,4058233,4058278,1,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_4-ephA,0,False,False,0,0,False,0,False,0,.,0,.


In [119]:
LPM_AllPep_V2_DF["WiNS_pGC_EventIDs"].value_counts().head(3)

WiNS_pGC_EventIDs
.            18597
Event_094       14
Event_096       11
Name: count, dtype: int64

In [120]:
LPM_AllPep_V2_DF["WiNS_mGC_EventIDs"].value_counts().head(3)

WiNS_mGC_EventIDs
.            18615
Event_094       14
Event_096       11
Name: count, dtype: int64

### Create DF of for all POSITIVE EPITOPES and NEGATIVE PEPTIDES (LPM)

In [121]:
LPM_AllPeptides_Mapped_DF.shape

(18741, 21)

In [122]:
LPM_PosEpi_V2_DF = LPM_AllPep_V2_DF.query("PosEpitope_Any == True")
LPM_PosEpi_V2_DF.shape

(424, 30)

In [123]:
LPM_NegPep_V2_DF = LPM_AllPep_V2_DF.query("PosEpitope_Any == False")
LPM_NegPep_V2_DF.shape

(18317, 30)

## Output TSVs of Epitopes (& Assayed peptides) annotated by the number of NS mutations (All NS mutations & NS mutation by mGCE)

In [124]:
Repo_Epitope_MainDir = "../../Data/220813_MtbEpitopes"

LPM_AllAssayedPeptides_WiMutInfo_TSV = f"{Repo_Epitope_MainDir}/240820.Panda24_Lind16.Merged.PeptidesMappedToRv.All.AnnoByMutFreq.V2.tsv" 

LPM_AllPep_V2_DF.to_csv(LPM_AllAssayedPeptides_WiMutInfo_TSV, sep ="\t", index=False)


In [125]:
!ls -1 $Repo_Epitope_MainDir/

220813.IEDB.MtbEpitopes.Filtered.Trim.tsv
220813.IEDB.MtbEpitopes.Filtered.tsv
240814.MtbEpitopes.2325.CuratedAndMerged.V1.tsv
240815.IEDB22.EpitopesMappedToRv.V1.tsv
240815.Lindestram2016.HLA_ResponseInfo.V1.tsv
240815.Lindestram2016.PeptidesMappedToRv.AllAssayed.V1.tsv
240815.Panda2024.PeptidesMappedToRv.AllAssayed.V1.tsv
240820.Panda24_Lind16.Merged.PeptidesMappedToRv.All.AnnoByMutFreq.V2.tsv
240820.Panda24_Lind16.Merged.PeptidesMappedToRv.AllAssayed.V1.tsv
240820.RvGene.EpitopeMappingStats.V1.tsv
Lindestam2016.PlosPatho.SuppData.Epitopes
Panda2024_EpitopeMapping


In [126]:
!wc -l $LPM_AllAssayedPeptides_WiMutInfo_TSV

18742 ../../Data/220813_MtbEpitopes/240820.Panda24_Lind16.Merged.PeptidesMappedToRv.All.AnnoByMutFreq.V2.tsv


# Part 2: Annotate each mutation by the number of epitopes overlapping

In [127]:
GubSNPs_CDSAnno_All_V2_DF.shape

(26508, 23)

In [128]:
GubSNPs_CDSAnno_All_V2_DF.head(3)

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,EventID,Pos_0based,Chrom,NumHmOvrlap,HmOvrlap,RegionType,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos,Ref_AA,Mut_AA,Start,End,MissenseMut,Lineage
0,1088,Node_1,N1176,G,A,NaN,1087,NC_000962.3,0,0,Unq,1,dnaA,+,1087.0,363.0,2.0,S,N,1087,1088,True,lineage5
1,10321,Node_1,N1176,C,T,NaN,10320,NC_000962.3,0,0,Unq,1,Rv0007,+,407.0,136.0,3.0,NaN,NaN,10320,10321,False,lineage5
2,11846,Node_1,N1176,C,G,NaN,11845,NC_000962.3,0,0,Unq,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11845,11846,False,lineage5


In [129]:
Epitope_CoordCols = ("Chrom", "Rv_Start", "Rv_End")

Gubbins_SNP_CoordCols = ("Chrom", "Start", "End")

Gub_SNPs_CDSEffect_WiEpiCt_DF = bf.count_overlaps(GubSNPs_CDSAnno_All_V2_DF,
                                                  LPM_PosEpi_V2_DF,      
                                                  cols1 = Gubbins_SNP_CoordCols,
                                                  cols2 = Epitope_CoordCols, ).rename(columns={'count': 'N_Epitopes_Overlap'})



gene_Num_NSMut_InEpitope_Dict = Gub_SNPs_CDSEffect_WiEpiCt_DF.set_index("Symbol")["N_Epitopes_Overlap"].count()

gene_Num_NSMutCausedBymGCE_Dict = Gub_SNPs_CDSEffect_WiEpiCt_DF.set_index("Symbol")["N_Epitopes_Overlap"].count()


### Output Gubbins SNPs annotated by CDS effect and Epitope overlap

In [130]:
Gub_SNPs_CDSEffect_WiEpiCt_DF.shape

(26508, 24)

In [131]:
Gubbins_SNPs_CDSAnno_WiEpitopeOverlap_TSV = f"{Gubbins_V1_OutputDir}/Gubbins.SNPs.AnnoByEvent.AnnoByCDSandEpitope.All.V2.tsv" 

Gub_SNPs_CDSEffect_WiEpiCt_DF.to_csv(Gubbins_SNPs_CDSAnno_WiEpitopeOverlap_TSV,
                                     sep ="\t",
                                     index=False)


In [132]:
!wc -l $Gubbins_SNPs_CDSAnno_WiEpitopeOverlap_TSV

26509 /n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V9/250901.WGA151CI.V9/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh/Gubbins.SNPs.AnnoByEvent.AnnoByCDSandEpitope.All.V2.tsv


# Part 3: Annotate each GC event by epitopes affected

In [133]:
mGCE_TrimCols = ['EventID', 'seqname', 'start_0based', 'end_1based',
                 'Parent_Node', 'Child_Node', 'Overlap_Genes',  'Top_KmerMatch_HomologGeneIDs',
                 'Max_KmerMatch_ToHm', 'NumHm_AtMaxKmerMatch', "Overlap_WiAntigenLVL2Gene"]

pGCE_V2_DF = pGCE_DF #[mGCE_TrimCols]


In [134]:
pGCE_V2_DF.head(1)


,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs,N_LCR_Ovrlap,OvrlapWi_LowComplexityRegion,N_LowPmap_Ovrlap,OvrlapWi_LowPmap,LenOfTaxaList,IsTermNode,Freq_SNP_FoundInAnyPR,NumHmTargets,NumHm_AtMaxKmerMatch,Top_KmerMatch_HomologTarIDs,Top_KmerMatch_HomologGeneIDs,Max_KmerMatch_ToHm,DistToHm_TopKmatch,TotalKmers_Eval,NumHm_AtMaxSNPMatch,Top_SNPMatch_HomologTarIDs,Top_SNPMatch_HomologGeneIDs,Max_SNPMatch_ToHm,DistToHm_TopSNPmatch,MappedEvent,Overlap_WiAntigenLVL2Gene
0,NC_000962.3,103600,104478,0.0,.,Node_148,Node_133,1737.432787,10,"['mada_2-31', 'mada_1-41', 'MT_0080', 'mada_10...",103599,104038.5,879,NaN,"Rv0093c,Rv0094c","Rv0093c,Rv0094c",False,False,True,False,Event_001,2,1,0,0,PR_HmRegion_002,0,0,1,1,130,False,0.0,2,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.8608,NaN,79,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.0,NaN,True,False


## Start processing (for-loop) to compare each event to all epitopes

In [135]:
V3_Rows = []

Dict_EventID_To_MutEpiID = {}

for i, row in tqdm(pGCE_V2_DF.iterrows() ):

    EventID = row["EventID"] 
    
    GCE_Start = row["start_0based"] 
    GCE_End = row["end_1based"] 
    # Define HmRegion range
    GCE_Range = f"NC_000962.3:{GCE_Start}-{GCE_End}"

    # 1) Count the number of AA changes caused by the event
    NS_Muts_InEvent_DF = Gub_SNPs_NSMutOnly_DF.query(f"EventID == '{EventID}'")

    row["N_NS_Mut"] = NS_Muts_InEvent_DF.shape[0]

    #print(EventID, NS_Muts_InEvent_DF.shape[0])


    # 2) Count the number of overlapping epitopes

    # 1) First get all pos epitopes which overlap with the region of interest
    Ovrlap_Epi_DF = bf.select(LPM_PosEpi_V2_DF,
                              GCE_Range,
                              cols=Epitope_CoordCols)

    if NS_Muts_InEvent_DF.shape[0] > 0:
        
        Ovrlap_Epi_DF = bf.count_overlaps(Ovrlap_Epi_DF,
                                          NS_Muts_InEvent_DF,
                                          cols1 = Epitope_CoordCols,
                                          cols2 = Gubbins_SNP_CoordCols).rename(columns={'count': 'N_Epi_ChangedByEvent'})

        Ovrlap_Epi_WiNSMut_DF = Ovrlap_Epi_DF.query("N_Epi_ChangedByEvent > 0")
        NumEpi_WiNSMut_ByEvent = Ovrlap_Epi_DF.query("N_Epi_ChangedByEvent > 0").shape[0]
        
        Ovrlap_EpiSym_IDs = list( Ovrlap_Epi_WiNSMut_DF["EpitopeSymbol_ID"].unique() )
        
    else:
        NumEpi_WiNSMut_ByEvent = 0
        Ovrlap_EpiSym_IDs = []
        Ovrlap_EpiSym_IDs = []

    # 1) First get all neg peptides which overlap with the region of interest
    Ovrlap_NegPep_DF = bf.select(LPM_NegPep_V2_DF, GCE_Range,
                                 cols=Epitope_CoordCols)

    if NS_Muts_InEvent_DF.shape[0] > 0:
        
        Ovrlap_Epi_DF = bf.count_overlaps(Ovrlap_NegPep_DF,
                                          NS_Muts_InEvent_DF,
                                          cols1 = Epitope_CoordCols,
                                          cols2 = Gubbins_SNP_CoordCols).rename(columns={'count': 'N_NegPep_ChangedByEvent'})

        
        Ovrlap_Epi_WiNSMut_DF = Ovrlap_Epi_DF.query("N_NegPep_ChangedByEvent > 0")
        NumNegPep_WiNSMut_ByEvent = Ovrlap_Epi_DF.query("N_NegPep_ChangedByEvent > 0").shape[0]
        
    else:
        NumNegPep_WiNSMut_ByEvent = 0
    
    Dict_EventID_To_MutEpiID[EventID] = Ovrlap_EpiSym_IDs

    row["N_EpiMutated_By_GCE"] = NumEpi_WiNSMut_ByEvent
    row["N_NegPepMutated_By_GCE"] = NumNegPep_WiNSMut_ByEvent
    row["EpitopeIDs_NSMutByEvent"] = ",".join(Ovrlap_EpiSym_IDs)

    V3_Rows.append(row)
    
pGCE_V3_DF = pd.DataFrame(V3_Rows)
pGCE_V3_DF["EpitopeIDs_NSMutByEvent"] = pGCE_V3_DF["EpitopeIDs_NSMutByEvent"].replace("", ".")

pGCE_V3_DF.shape

324it [00:08, 38.03it/s]


(324, 51)

#### Peak at pGCE DF annotated by overlap w/ annotated epitopes

In [136]:
pGCE_V3_DF["EpitopeIDs_NSMutByEvent"].value_counts().head(4)

EpitopeIDs_NSMutByEvent
.                                               299
UnqPeptide_7795-PPE60                             5
UnqPeptide_11315-esxL                             4
UnqPeptide_1141-PPE18,UnqPeptide_16789-PPE18      3
Name: count, dtype: int64

In [137]:
pGCE_V3_DF.sort_values("N_EpiMutated_By_GCE", ascending=False).head(5)

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs,N_LCR_Ovrlap,OvrlapWi_LowComplexityRegion,N_LowPmap_Ovrlap,OvrlapWi_LowPmap,LenOfTaxaList,IsTermNode,Freq_SNP_FoundInAnyPR,NumHmTargets,NumHm_AtMaxKmerMatch,Top_KmerMatch_HomologTarIDs,Top_KmerMatch_HomologGeneIDs,Max_KmerMatch_ToHm,DistToHm_TopKmatch,TotalKmers_Eval,NumHm_AtMaxSNPMatch,Top_SNPMatch_HomologTarIDs,Top_SNPMatch_HomologGeneIDs,Max_SNPMatch_ToHm,DistToHm_TopSNPmatch,MappedEvent,Overlap_WiAntigenLVL2Gene,N_NS_Mut,N_EpiMutated_By_GCE,N_NegPepMutated_By_GCE,EpitopeIDs_NSMutByEvent
93,NC_000962.3,1339648,1339806,0.0,.,Node_25,M0017522_5,1625.533450,20,['M0017522_5'],1339647,1339726.5,159,lineage4,PPE18,Rv1196,False,True,False,False,Event_094,2,1,0,0,PR_HmRegion_053,0,0,1,1,1,True,1.000000,2,1,"PE31,PPE60-H37Rv-3894016-3895519","PE31,PPE60",1.0000,1856491.0,114,1,"PE31,PPE60-H37Rv-3894016-3895519","PE31,PPE60",1.000,1856491.0,True,True,17,8,6,"UnqPeptide_1812-PPE18,UnqPeptide_5494-PPE18,Un..."
113,NC_000962.3,1533327,1533434,0.0,.,Node_30,Node_29,276.408286,4,"['TB2661', 'TB3386']",1533326,1533380.0,108,lineage4,PPE19,Rv1361c,False,True,False,False,Event_114,2,1,0,0,PR_HmRegion_060,0,0,1,1,2,False,1.000000,2,1,PPE60-H37Rv-3894405-3895588,PPE60,1.0000,2049915.5,40,1,PPE60-H37Rv-3894405-3895588,PPE60,0.250,2049915.5,True,True,3,5,5,"UnqPeptide_4240-PPE19,UnqPeptide_5681-PPE19,Un..."
94,NC_000962.3,1339861,1339905,0.0,.,Node_43,TB3237,688.268204,12,['TB3237'],1339860,1339882.5,45,lineage4,PPE18,Rv1196,False,True,False,False,Event_095,2,1,0,0,PR_HmRegion_053,0,0,1,1,1,True,1.000000,2,2,"PE31,PPE60-H37Rv-3894016-3895519-PPE19-H37Rv-1...","PE31,PPE60-PPE19",1.0000,NaN,55,1,"PE31,PPE60-H37Rv-3894016-3895519","PE31,PPE60",1.000,1856647.0,True,True,8,4,3,"UnqPeptide_998-PPE18,UnqPeptide_8744-PPE18,Unq..."
95,NC_000962.3,1339894,1340208,0.0,.,Node_17,TB3334,993.431569,8,['TB3334'],1339893,1340050.5,315,lineage4,PPE18,Rv1196,False,True,False,False,Event_096,2,1,0,0,PR_HmRegion_053,0,0,3,1,1,True,1.000000,2,1,PPE19-H37Rv-1532539-1533632,PPE19,0.9688,193035.0,64,1,"PE31,PPE60-H37Rv-3894016-3895519","PE31,PPE60",0.500,1856815.0,True,True,7,3,12,"UnqPeptide_959-PPE18,UnqPeptide_998-PPE18,UnqP..."
208,NC_000962.3,2626004,2626600,0.0,.,Node_149,RW-TB008,4814.594574,24,['RW-TB008'],2626003,2626301.5,597,lineage8,"esxO,esxP","Rv2346c,Rv2347c",True,False,False,False,Event_209,4,1,0,0,PR_HmRegion_113,0,0,1,1,1,True,0.541667,4,1,"PPE18,esxK,esxL-H37Rv-1340494-1341292","PPE18,esxK,esxL",0.5896,1285408.5,173,1,"esxI,esxJ-H37Rv-1160545-1161167","esxI,esxJ",0.375,1465445.5,True,True,6,3,1,"UnqPeptide_4422-esxP,UnqPeptide_9768-esxO,UnqP..."


In [138]:
pGCE_V3_DF.sort_values("N_EpiMutated_By_GCE", ascending=False)["EpitopeIDs_NSMutByEvent"].values[0]

'UnqPeptide_1812-PPE18,UnqPeptide_5494-PPE18,UnqPeptide_7207-PPE18,UnqPeptide_8677-PPE18,UnqPeptide_15834-PPE18,UnqPeptide_16444-PPE18,UnqPeptide_17231-PPE18,UnqPeptide_17390-PPE18'

In [139]:
pGCE_V3_DF.head(2)

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs,N_LCR_Ovrlap,OvrlapWi_LowComplexityRegion,N_LowPmap_Ovrlap,OvrlapWi_LowPmap,LenOfTaxaList,IsTermNode,Freq_SNP_FoundInAnyPR,NumHmTargets,NumHm_AtMaxKmerMatch,Top_KmerMatch_HomologTarIDs,Top_KmerMatch_HomologGeneIDs,Max_KmerMatch_ToHm,DistToHm_TopKmatch,TotalKmers_Eval,NumHm_AtMaxSNPMatch,Top_SNPMatch_HomologTarIDs,Top_SNPMatch_HomologGeneIDs,Max_SNPMatch_ToHm,DistToHm_TopSNPmatch,MappedEvent,Overlap_WiAntigenLVL2Gene,N_NS_Mut,N_EpiMutated_By_GCE,N_NegPepMutated_By_GCE,EpitopeIDs_NSMutByEvent
0,NC_000962.3,103600,104478,0.0,.,Node_148,Node_133,1737.432787,10,"['mada_2-31', 'mada_1-41', 'MT_0080', 'mada_10...",103599,104038.5,879,NaN,"Rv0093c,Rv0094c","Rv0093c,Rv0094c",False,False,True,False,Event_001,2,1,0,0,PR_HmRegion_002,0,0,1,1,130,False,0.0,2,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.8608,NaN,79,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.0,NaN,True,False,0,0,0,.
1,NC_000962.3,103676,103930,0.0,.,Node_3,N1177,1950.228311,5,['N1177'],103675,103802.5,255,lineage6,Rv0094c,Rv0094c,False,False,True,False,Event_002,2,1,0,0,PR_HmRegion_002,0,0,1,1,1,True,0.0,2,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.0000,NaN,46,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.0,NaN,False,False,4,0,0,.


In [140]:
pGCE_V3_DF["EpitopeIDs_NSMutByEvent"].dtype

dtype('O')

In [141]:
pGCE_V3_DF["EpitopeIDs_NSMutByEvent"].dtype

dtype('O')

In [142]:
type( pGCE_V3_DF["EpitopeIDs_NSMutByEvent"].values[0] )

str

In [143]:
pGCE_V3_DF["EpitopeIDs_NSMutByEvent"].values[0:5]

array(['.', '.', '.', '.', '.'], dtype=object)

In [144]:
pGCE_V3_DF.shape

(324, 51)

In [145]:
LPM_AllPep_V2_DF.head(1)

,Epitope_ID,Epitope_Seq,Epitope_Len,RvID,Symbol,AA_Start,AA_End,Chrom,Rv_Start,Rv_End,EpitopeSeqFreqInAntigen,Dataset,Assayed_Panda24,PosEpitope_Panda24,Assayed_Lindestam16,PosEpitope_Lindestam16,PosEpitope_Any,EpitopeSymbol_ID,N_HmRegion,HasHmRegion,Antigen_LVL2,N_NSMut_Total,N_NSMut_mGCE,IsMutBymGCE,N_NSMut_pGCE,IsMutBypGCE,N_mGCEs_WiNS,WiNS_mGC_EventIDs,N_pGCEs_WiNS,WiNS_pGC_EventIDs
0,UnqPeptide_1,FPTLNYAVSVAEACE,15,Rv0322,udgA,368,383,NC_000962.3,390363,390408,1,Lindestam16_Panda24_Merged,True,False,False,False,False,UnqPeptide_1-udgA,0,False,False,0,0,False,0,False,0,.,0,.


In [146]:
LPM_AllPep_V2_DF.query("EpitopeSymbol_ID == 'UnqPeptide_14059-PPE60' ")

,Epitope_ID,Epitope_Seq,Epitope_Len,RvID,Symbol,AA_Start,AA_End,Chrom,Rv_Start,Rv_End,EpitopeSeqFreqInAntigen,Dataset,Assayed_Panda24,PosEpitope_Panda24,Assayed_Lindestam16,PosEpitope_Lindestam16,PosEpitope_Any,EpitopeSymbol_ID,N_HmRegion,HasHmRegion,Antigen_LVL2,N_NSMut_Total,N_NSMut_mGCE,IsMutBymGCE,N_NSMut_pGCE,IsMutBypGCE,N_mGCEs_WiNS,WiNS_mGC_EventIDs,N_pGCEs_WiNS,WiNS_pGC_EventIDs


In [147]:
LPM_AllPep_V2_DF.query("Epitope_Seq == 'LGGLWTAVSPHLSPL' ")

,Epitope_ID,Epitope_Seq,Epitope_Len,RvID,Symbol,AA_Start,AA_End,Chrom,Rv_Start,Rv_End,EpitopeSeqFreqInAntigen,Dataset,Assayed_Panda24,PosEpitope_Panda24,Assayed_Lindestam16,PosEpitope_Lindestam16,PosEpitope_Any,EpitopeSymbol_ID,N_HmRegion,HasHmRegion,Antigen_LVL2,N_NSMut_Total,N_NSMut_mGCE,IsMutBymGCE,N_NSMut_pGCE,IsMutBypGCE,N_mGCEs_WiNS,WiNS_mGC_EventIDs,N_pGCEs_WiNS,WiNS_pGC_EventIDs
7810,UnqPeptide_7795,LGGLWTAVSPHLSPL,15,Rv3478,PPE60,223,238,NC_000962.3,3895094,3895139,1,Lindestam16_Panda24_Merged,True,False,True,True,True,UnqPeptide_7795-PPE60,1,True,True,24,24,True,24,True,5,"Event_295,Event_291,Event_290,Event_292,Event_293",5,"Event_295,Event_291,Event_290,Event_292,Event_293"


In [148]:
Lind16_EpiPosFreq_DF.head(1)

,Epitope_Seq,Num_Positive,Num_Assayed,Fraction_Positive,Gene(s)
0,AAAQASAAAAAYEAA,0,63,0.0,PPE30


In [149]:
Lind16_EpiPosFreq_DF.query("Epitope_Seq == 'LGGLWTAVSPHLSPL' ")

,Epitope_Seq,Num_Positive,Num_Assayed,Fraction_Positive,Gene(s)
395,LGGLWTAVSPHLSPL,2,63,0.031746,PPE60


## Output mGC events annotated by match to paralogs + epitope mutations

In [150]:

RecombPreds_Anno_ByHmMatch_ByEpiMut_V3_TSV = f"{Gubbins_V1_OutputDir}/Gubbins.All_GCEs.AnnoBy.EpitopeEffect.V3.tsv"

pGCE_V3_DF.to_csv(RecombPreds_Anno_ByHmMatch_ByEpiMut_V3_TSV, sep ="\t", index=False)

In [151]:
!ls -lah $RecombPreds_Anno_ByHmMatch_ByEpiMut_V3_TSV

-rw-r--r-- 1 mm774 hpc_farhat 137K Sep 24 14:47 /n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V9/250901.WGA151CI.V9/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh/Gubbins.All_GCEs.AnnoBy.EpitopeEffect.V3.tsv


# Part 4: Produce gene-level summary of mapped epitopes and mutational burden

## A) Subset Gubbins ASR SNPs (V2 Table) into groups of interest

In [152]:
Gub_SNPs_V2_DF                 = Gub_SNPs_CDSEffect_WiEpiCt_DF

# Gub_SNPs_V2_WiEpiMut_DF        = Gub_SNPs_V2_DF.query(" N_Epitopes_Overlap > 0 ")

Gub_SNPs_V2_NSMutOnly_DF       = Gub_SNPs_V2_DF[~Gub_SNPs_V2_DF["Mut_AA"].isna()].query("Ref_AA != Mut_AA")

Gub_SNPs_V2_NSMutIn_pGCE_DF    = Gub_SNPs_V2_NSMutOnly_DF.query("EventID != 'None'")

Gub_SNPs_V2_NSMutIn_mGCE_DF    = Gub_SNPs_V2_NSMutOnly_DF[ Gub_SNPs_V2_NSMutOnly_DF["EventID"].isin(mGCE_EventIDs) ]

Gub_SNPs_V2_NSMutInEpitope_DF  = Gub_SNPs_V2_NSMutOnly_DF.query(" N_Epitopes_Overlap > 0 ")


Gub_SNPs_V2_NSMutIn_pGCE_InEpitope_DF  = Gub_SNPs_V2_NSMutIn_pGCE_DF.query(" N_Epitopes_Overlap > 0 ")

Gub_SNPs_V2_NSMutIn_mGCE_InEpitope_DF  = Gub_SNPs_V2_NSMutIn_mGCE_DF.query(" N_Epitopes_Overlap > 0 ")


## B) Calculate missense mutation freq stats per gene

#### Count the number of Non-synonymous mutations in all annotated genes

In [153]:
gene_Num_All_NSMuts_sum = (Gub_SNPs_V2_NSMutOnly_DF.groupby('Symbol')['Child_Node'].count()
    .sort_values(ascending=False)
    .reset_index()
)

gene_Num_All_NSMuts_sum_Dict = gene_Num_All_NSMuts_sum.set_index("Symbol")["Child_Node"].to_dict()


In [154]:
gene_Num_All_NSMuts_sum_Dict["PPE18"]

61

#### Count the number of NS mutations caused by a **mapped** GC event (across all genes)

In [155]:
gene_Num_mGCE_NSMuts_sum = (Gub_SNPs_V2_NSMutIn_mGCE_DF.query("Ref_AA != Mut_AA").groupby('Symbol')['Child_Node'].count()
    .sort_values(ascending=False).reset_index()
)

gene_Num_mGCE_NSMuts_sum_Dict = gene_Num_mGCE_NSMuts_sum.set_index("Symbol")["Child_Node"].to_dict()


In [156]:
gene_Num_mGCE_NSMuts_sum_Dict["PPE18"]

45

In [157]:
gene_Num_mGCE_NSMuts_sum_Dict["PPE60"]

120

In [158]:
gene_Num_mGCE_NSMuts_sum_Dict["PPE19"]

36

#### Count the number of NS mutations caused by a **putative** GC event (across all genes)

In [159]:
gene_Num_pGCE_NSMuts_sum = (Gub_SNPs_NSMutIn_pGCE_DF.query("Ref_AA != Mut_AA").groupby('Symbol')['Child_Node'].count()
    .sort_values(ascending=False)
    .reset_index()
)

gene_Num_pGCE_NSMuts_sum_Dict = gene_Num_pGCE_NSMuts_sum.set_index("Symbol")["Child_Node"].to_dict()


In [160]:
gene_Num_pGCE_NSMuts_sum_Dict["PPE18"]

46

In [161]:
gene_Num_pGCE_NSMuts_sum_Dict["PPE60"]

120

In [162]:
gene_Num_pGCE_NSMuts_sum_Dict["PPE19"]

36

In [163]:
gene_Num_pGCE_NSMuts_sum_Dict["PPE46"]

11

#### Let's quantify the number of NS Mut Events overlapping w/ an epitope

In [164]:

gene_Num_NSMutsInEpitopes_sum = (Gub_SNPs_V2_NSMutInEpitope_DF.groupby('Symbol')['N_Epitopes_Overlap'].count()
                                    .sort_values(ascending=False).reset_index())  

gene_Num_NSMutInEpitope_Dict = gene_Num_NSMutsInEpitopes_sum.set_index("Symbol")["N_Epitopes_Overlap"].to_dict()  


#### Let's quantify the number of NS Mut Events overlapping w/ an epitope & part of a MAPPED GC EVENT

In [165]:

gene_Num_NSMutsInEpitopes_In_mGCE_sum = (Gub_SNPs_V2_NSMutIn_mGCE_DF.groupby('Symbol')['N_Epitopes_Overlap'].count()
                                    .sort_values(ascending=False).reset_index())  

gene_Num_NSMutInEpitope_In_mGCE_Dict = gene_Num_NSMutsInEpitopes_In_mGCE_sum.set_index("Symbol")["N_Epitopes_Overlap"].to_dict()  


#### Let's quantify the number of NS Mut Events overlapping w/ an epitope & part of a PUTATIVE GC EVENT

In [166]:

gene_Num_NSMutsInEpitopes_In_pGCE_sum = (Gub_SNPs_V2_NSMutIn_pGCE_DF.groupby('Symbol')['N_Epitopes_Overlap'].count()
                                    .sort_values(ascending=False).reset_index())  

gene_Num_NSMutInEpitope_In_pGCE_Dict = gene_Num_NSMutsInEpitopes_In_pGCE_sum.set_index("Symbol")["N_Epitopes_Overlap"].to_dict()  


In [167]:
gene_Num_NSMutInEpitope_In_pGCE_Dict == gene_Num_NSMutInEpitope_In_mGCE_Dict

False

## C) Create Gene-Level Summary DF (ALL ANNOTATED CDS genes)

In [168]:
mGCE_V3_DF = pGCE_V3_DF.query("(Max_KmerMatch_ToHm > 0.5)")
mGCE_V3_DF.shape

(213, 51)

In [169]:
Genes_GCandMutFreqStats_DF = bf.count_overlaps(Rv_Genes_EpitopeSummary_DF, pGCE_V3_DF,
                                               cols1 = GenomeAnno_CoordCols,
                                               cols2 = RE_CoordCols ).rename(columns={'count': 'pGCE_Ovrlap'})


Genes_GCandMutFreqStats_DF = bf.count_overlaps(Genes_GCandMutFreqStats_DF, mGCE_V3_DF,
                                                 cols1 = GenomeAnno_CoordCols,
                                               cols2 = RE_CoordCols ).rename(columns={'count': 'mGCE_Ovrlap'})

Genes_GCandMutFreqStats_DF["Total_NS_Muts"] = Genes_GCandMutFreqStats_DF["Symbol"].map(gene_Num_All_NSMuts_sum_Dict).fillna(0)
Genes_GCandMutFreqStats_DF["Total_NS_Muts_InEpitope"] = Genes_GCandMutFreqStats_DF["Symbol"].map(gene_Num_NSMutInEpitope_Dict).fillna(0)
Genes_GCandMutFreqStats_DF["Total_NS_Muts_BymGCE"] = Genes_GCandMutFreqStats_DF["Symbol"].map(gene_Num_mGCE_NSMuts_sum_Dict).fillna(0)
Genes_GCandMutFreqStats_DF["Total_NS_Muts_BypGCE"] = Genes_GCandMutFreqStats_DF["Symbol"].map(gene_Num_pGCE_NSMuts_sum_Dict).fillna(0)

Genes_GCandMutFreqStats_DF["Total_NS_Muts_BymGCE_InEpitope"] = Genes_GCandMutFreqStats_DF["Symbol"].map(gene_Num_NSMutInEpitope_In_mGCE_Dict).fillna(0)
Genes_GCandMutFreqStats_DF["Total_NS_Muts_BypGCE_InEpitope"] = Genes_GCandMutFreqStats_DF["Symbol"].map(gene_Num_NSMutInEpitope_In_pGCE_Dict).fillna(0)


Genes_GCandMutFreqStats_DF["RelFreq_NS_Muts"]          = ( Genes_GCandMutFreqStats_DF["Total_NS_Muts"] / Genes_GCandMutFreqStats_DF["Length"] ) * 300
Genes_GCandMutFreqStats_DF["RelFreq_NS_Mut_InEpitope"] = ( Genes_GCandMutFreqStats_DF["Total_NS_Muts_InEpitope"] / Genes_GCandMutFreqStats_DF["Length"] ) * 300   


Genes_GCandMutFreqStats_DF["Has_mGCE"]                   = Genes_GCandMutFreqStats_DF["mGCE_Ovrlap"] > 0 
Genes_GCandMutFreqStats_DF["Has_mGCE_WiNSMut"]           = Genes_GCandMutFreqStats_DF["Total_NS_Muts_BymGCE"] > 0 
Genes_GCandMutFreqStats_DF["Has_mGCE_WiNSMut_InEpitope"] = Genes_GCandMutFreqStats_DF["Total_NS_Muts_BymGCE_InEpitope"] > 0 

Genes_GCandMutFreqStats_DF["Has_pGCE"]                   = Genes_GCandMutFreqStats_DF["pGCE_Ovrlap"] > 0 
Genes_GCandMutFreqStats_DF["Has_pGCE_WiNSMut"]           = Genes_GCandMutFreqStats_DF["Total_NS_Muts_BypGCE"] > 0 
Genes_GCandMutFreqStats_DF["Has_pGCE_WiNSMut_InEpitope"] = Genes_GCandMutFreqStats_DF["Total_NS_Muts_BypGCE_InEpitope"] > 0 


Genes_GCandMutFreqStats_DF.shape

(3841, 36)

In [170]:
Genes_GCandMutFreqStats_DF.head(4)

,Chrom,Start,End,Strand,Feature,H37rv_GeneID,Symbol,Functional_Category,Gene_Cat_V2,N_Pos,N_Neg,Total,Positive_Proportion,Middle,Length,AnyPosEpitope,Antigen_LVL2,N_HmRegion,HasHmRegion,AntigenLVL2_And_HHR_Comb,pGCE_Ovrlap,mGCE_Ovrlap,Total_NS_Muts,Total_NS_Muts_InEpitope,Total_NS_Muts_BymGCE,Total_NS_Muts_BypGCE,Total_NS_Muts_BymGCE_InEpitope,Total_NS_Muts_BypGCE_InEpitope,RelFreq_NS_Muts,RelFreq_NS_Mut_InEpitope,Has_mGCE,Has_mGCE_WiNSMut,Has_mGCE_WiNSMut_InEpitope,Has_pGCE,Has_pGCE_WiNSMut,Has_pGCE_WiNSMut_InEpitope
0,NC_000962.3,0,1524,+,CDS,Rv0001,dnaA,information pathways,information pathways,0,8,8,0.0,762.0,1524,False,False,0,False,NonReactive-Unq,0,0,10.0,0.0,0.0,0.0,0.0,10.0,1.968504,0.0,False,False,False,False,False,True
1,NC_000962.3,2051,3260,+,CDS,Rv0002,dnaN,information pathways,information pathways,0,6,6,0.0,2655.5,1209,False,False,0,False,NonReactive-Unq,0,0,2.0,0.0,0.0,0.0,0.0,2.0,0.496278,0.0,False,False,False,False,False,True
2,NC_000962.3,3279,4437,+,CDS,Rv0003,recF,information pathways,information pathways,0,6,6,0.0,3858.0,1158,False,False,0,False,NonReactive-Unq,0,0,7.0,0.0,0.0,0.0,0.0,7.0,1.813472,0.0,False,False,False,False,False,True
3,NC_000962.3,4433,4997,+,CDS,Rv0004,Rv0004,conserved hypotheticals,conserved hypotheticals,0,3,3,0.0,4715.0,564,False,False,0,False,NonReactive-Unq,0,0,5.0,0.0,0.0,0.0,0.0,5.0,2.659574,0.0,False,False,False,False,False,True


## Output Mutational Burden Stats summarized across Rv genes + antigens

In [171]:
Repo_AntigenMut_MainDir = "../../Data/250609.AntigenMutationalBurdenAnalysis"
!mkdir $Repo_AntigenMut_MainDir

Rv_Gene_MutStats_V1_TSV = f"{Repo_AntigenMut_MainDir}/250808.RvGenes.GCE.MutationFreq.Epitopes.SummaryStats.V1.tsv"


mkdir: cannot create directory ‘../../Data/250609.AntigenMutationalBurdenAnalysis’: File exists


In [172]:
Genes_GCandMutFreqStats_DF.to_csv(Rv_Gene_MutStats_V1_TSV,
                                  sep = "\t",
                                  index=False)

Genes_GCandMutFreqStats_DF.shape

(3841, 36)

# Part 5: Generate mutation and epitope summary stats per genome and protein codon position

## A) Make H37Rv per-position reference DF

In [173]:
def MakeRefGenomeDF(chrom_Length: int, chrom_Name: str):
    """
    Generates a Pandas DataFrame representing genomic positions for a given chromosome.

    Parameters:
    chrom_Length (int): The length of the chromosome (number of base pairs).
    chrom_Name (str): The name of the chromosome.

    Returns:
    pd.DataFrame: A DataFrame with columns 'chrom', 'start', and 'end'.
    """
    return pd.DataFrame({
        "chrom": [chrom_Name] * chrom_Length,
        "start": range(chrom_Length),
        "end": range(1, chrom_Length + 1)
    })

In [174]:
Rv_AllPos_DF = MakeRefGenomeDF(4411532, "NC_000962.3")
Rv_AllPos_DF.shape

(4411532, 3)

In [175]:
Rv_AllPos_DF.head(3)

,chrom,start,end
0,NC_000962.3,0,1
1,NC_000962.3,1,2
2,NC_000962.3,2,3


In [176]:
Rv_AllPos_DF.tail(3)

,chrom,start,end
4411529,NC_000962.3,4411529,4411530
4411530,NC_000962.3,4411530,4411531
4411531,NC_000962.3,4411531,4411532


## B) Create DF of per-codon mutation frequency 

In [177]:
Gub_SNPs_V2_NSMutOnly_DF.head(1)

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,EventID,Pos_0based,Chrom,NumHmOvrlap,HmOvrlap,RegionType,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos,Ref_AA,Mut_AA,Start,End,MissenseMut,Lineage,N_Epitopes_Overlap
0,1088,Node_1,N1176,G,A,NaN,1087,NC_000962.3,0,0,Unq,1,dnaA,+,1087.0,363.0,2.0,S,N,1087,1088,True,lineage5,0


In [178]:
Gub_SNPs_V2_NSMutOnly_DF.shape

(13883, 24)

In [179]:
# Group by gene symbol and codon position, then count occurrences
RvPerCodon_NSMutFreq_DF = (
    Gub_SNPs_NSMutOnly_DF
    .groupby(["Symbol", "Codon"])
    .agg(Mutation_Count=("Codon", "count"), Pos_0based=("Pos_0based", "min"))
    .reset_index()
)

In [180]:
RvPerCodon_NSMutFreq_DF.head(5)

,Symbol,Codon,Mutation_Count,Pos_0based
0,35kd_ag,230.0,1,3057374
1,35kd_ag,252.0,1,3057308
2,35kd_ag,270.0,1,3057255
3,PE1,122.0,1,178945
4,PE1,123.0,1,178940


In [181]:
#PerCodon_MutCt_DF = PerCodon_NSMutFreq_DF

In [182]:
RvPerCodon_NSMutFreq_DF.query("Symbol == 'PPE18' ").head(4)

,Symbol,Codon,Mutation_Count,Pos_0based
839,PPE18,30.0,4,1339435
840,PPE18,32.0,1,1339443
841,PPE18,55.0,2,1339510
842,PPE18,67.0,1,1339547


## C) Create a DF of epitope coverage and assayed peptide coverage per H37Rv position

In [183]:
Epitope_CoordCols = ("Chrom", "Rv_Start", "Rv_End")

Rv_EpitopeCov_DF = bf.count_overlaps(Rv_AllPos_DF,
                                     LPM_PosEpi_V2_DF,      
                                     cols2 = Epitope_CoordCols, ).rename(columns={'count': 'Cov_PosEpitopes'})

Rv_EpitopeCov_DF = bf.count_overlaps(Rv_EpitopeCov_DF,
                                     LPM_AllPep_V2_DF,      
                                     cols2 = Epitope_CoordCols, ).rename(columns={'count': 'Cov_AssayedPeptides'})


Rv_EpitopeCov_DF.shape

(4411532, 5)

In [184]:
Rv_EpitopeCov_DF.head()

,chrom,start,end,Cov_PosEpitopes,Cov_AssayedPeptides
0,NC_000962.3,0,1,0,0
1,NC_000962.3,1,2,0,0
2,NC_000962.3,2,3,0,0
3,NC_000962.3,3,4,0,0
4,NC_000962.3,4,5,0,0


In [185]:
Rv_EpitopeCov_NonZero_DF = Rv_EpitopeCov_DF.query(" Cov_AssayedPeptides > 0")
Rv_EpitopeCov_NonZero_DF.shape

(746901, 5)

## E) Output Rv position level stats (Epitope mapping and Codon mutation)

In [186]:
Repo_AntigenMut_MainDir = "../../Data/250609.AntigenMutationalBurdenAnalysis"
!mkdir $Repo_AntigenMut_MainDir
PerRvPosition_EpitopeCovStats_TSV = f"{Repo_AntigenMut_MainDir}/250609.PerRvPos.EpitopeCoverage.AssayedPosOnly.V1.tsv.gz"

PerRvCodon_MutationFreq_TSV = f"{Repo_AntigenMut_MainDir}/250609.PerRvCodon.MissenseMutatationFreq.V1.tsv.gz"


mkdir: cannot create directory ‘../../Data/250609.AntigenMutationalBurdenAnalysis’: File exists


In [187]:
Rv_EpitopeCov_NonZero_DF.to_csv(PerRvPosition_EpitopeCovStats_TSV,
                                  sep = "\t",
                                  index=False)

RvPerCodon_NSMutFreq_DF.to_csv(PerRvCodon_MutationFreq_TSV,
                                  sep = "\t",
                                  index=False)

In [188]:
!ls -lah $PerRvPosition_EpitopeCovStats_TSV

-rw-r--r-- 1 mm774 hpc_farhat 3.6M Sep 24 14:47 ../../Data/250609.AntigenMutationalBurdenAnalysis/250609.PerRvPos.EpitopeCoverage.AssayedPosOnly.V1.tsv.gz


In [189]:
!ls -lah $PerRvCodon_MutationFreq_TSV

-rw-r--r-- 1 mm774 hpc_farhat 86K Sep 24 14:47 ../../Data/250609.AntigenMutationalBurdenAnalysis/250609.PerRvCodon.MissenseMutatationFreq.V1.tsv.gz


In [190]:
Genes_GCandMutFreqStats_DF.head()

,Chrom,Start,End,Strand,Feature,H37rv_GeneID,Symbol,Functional_Category,Gene_Cat_V2,N_Pos,N_Neg,Total,Positive_Proportion,Middle,Length,AnyPosEpitope,Antigen_LVL2,N_HmRegion,HasHmRegion,AntigenLVL2_And_HHR_Comb,pGCE_Ovrlap,mGCE_Ovrlap,Total_NS_Muts,Total_NS_Muts_InEpitope,Total_NS_Muts_BymGCE,Total_NS_Muts_BypGCE,Total_NS_Muts_BymGCE_InEpitope,Total_NS_Muts_BypGCE_InEpitope,RelFreq_NS_Muts,RelFreq_NS_Mut_InEpitope,Has_mGCE,Has_mGCE_WiNSMut,Has_mGCE_WiNSMut_InEpitope,Has_pGCE,Has_pGCE_WiNSMut,Has_pGCE_WiNSMut_InEpitope
0,NC_000962.3,0,1524,+,CDS,Rv0001,dnaA,information pathways,information pathways,0,8,8,0.0,762.0,1524,False,False,0,False,NonReactive-Unq,0,0,10.0,0.0,0.0,0.0,0.0,10.0,1.968504,0.0,False,False,False,False,False,True
1,NC_000962.3,2051,3260,+,CDS,Rv0002,dnaN,information pathways,information pathways,0,6,6,0.0,2655.5,1209,False,False,0,False,NonReactive-Unq,0,0,2.0,0.0,0.0,0.0,0.0,2.0,0.496278,0.0,False,False,False,False,False,True
2,NC_000962.3,3279,4437,+,CDS,Rv0003,recF,information pathways,information pathways,0,6,6,0.0,3858.0,1158,False,False,0,False,NonReactive-Unq,0,0,7.0,0.0,0.0,0.0,0.0,7.0,1.813472,0.0,False,False,False,False,False,True
3,NC_000962.3,4433,4997,+,CDS,Rv0004,Rv0004,conserved hypotheticals,conserved hypotheticals,0,3,3,0.0,4715.0,564,False,False,0,False,NonReactive-Unq,0,0,5.0,0.0,0.0,0.0,0.0,5.0,2.659574,0.0,False,False,False,False,False,True
4,NC_000962.3,5239,7267,+,CDS,Rv0005,gyrB,information pathways,information pathways,0,8,8,0.0,6253.0,2028,False,False,0,False,NonReactive-Unq,0,0,11.0,0.0,0.0,0.0,0.0,11.0,1.627219,0.0,False,False,False,False,False,True


# Part 6) Create DF of NS Mutation stats across all 53 T-cell antigens (`LVL2`)

In [191]:
Antigen_GCandMutFreqStats_DF = Genes_GCandMutFreqStats_DF.query("Antigen_LVL2 == True")

Antigen_GCandMutFreqStats_DF = Antigen_GCandMutFreqStats_DF.sort_values("RelFreq_NS_Muts", ascending=False)

#### Calculate the proportion of each antigen that is covered by at least 1 epitope

listOf_LenPartOfEpitopes_Antigens = []

for i, row in Antigen_GCandMutFreqStats_DF.iterrows():
    
    i_Start, i_End = row["Start"], row["End"]
    i_Reg_EpiCov_DF = Rv_EpitopeCov_NonZero_DF.query(f"start >= {i_Start} & end <= {i_End} ")

    i_Length_PartOfEpitope = i_Reg_EpiCov_DF.query("Cov_PosEpitopes >= 1").shape[0]
    listOf_LenPartOfEpitopes_Antigens.append(i_Length_PartOfEpitope)

Antigen_GCandMutFreqStats_DF["Length_WiPosEpitope"] = listOf_LenPartOfEpitopes_Antigens
Antigen_GCandMutFreqStats_DF["RelLength_WiPosEpitope"] = Antigen_GCandMutFreqStats_DF["Length_WiPosEpitope"] / Antigen_GCandMutFreqStats_DF["Length"]
#Antigen_NSMutInfo_DF["RelFreq_NS_Mut_NormalizedByEpitopeLength"] = ( Antigen_NSMutInfo_DF["Total_NS_Muts_InEpitope"] / Antigen_NSMutInfo_DF["Length_WiPosEpitope"] ) * 300

Antigen_GCandMutFreqStats_DF.shape


(53, 38)

## Output Mutational Burden Stats summarized across ONLY antigens (L2)

In [192]:
Repo_AntigenMut_MainDir = "../../Data/250609.AntigenMutationalBurdenAnalysis"
!mkdir $Repo_AntigenMut_MainDir

Rv_Antigens_MutStats_V1_TSV = f"{Repo_AntigenMut_MainDir}/250808.RvL2Antigens.GCE.MutationFreq.Epitopes.SummaryStats.V1.tsv"


mkdir: cannot create directory ‘../../Data/250609.AntigenMutationalBurdenAnalysis’: File exists


In [193]:
Antigen_GCandMutFreqStats_DF.to_csv(Rv_Antigens_MutStats_V1_TSV,
                                  sep = "\t",
                                  index=False)

Antigen_GCandMutFreqStats_DF.shape

(53, 38)

# Test reparsing of all  processed data TSVs output by this notebook

### a) Parse table of putative GCEs annotated by epitope mutation and overlap

In [194]:
RecombPreds_Anno_ByHmMatch_ByEpiMut_V3_TSV = f"{Gubbins_V1_OutputDir}/Gubbins.All_GCEs.AnnoBy.EpitopeEffect.V3.tsv"


In [195]:
pGCE_V3_DF = pd.read_csv(RecombPreds_Anno_ByHmMatch_ByEpiMut_V3_TSV, sep ="\t")
pGCE_V3_DF.shape

(324, 51)

### b) Parse table of SNP Events annotated by CDS effect + epitope overlap

In [196]:
Gubbins_V1_OutputDir = f"{Target_Output_Dir}/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh"

Gubbins_OutPrefix = "Gubbins"
Gubbins_Prefix_PATH = f"{Gubbins_V1_OutputDir}/{Gubbins_OutPrefix}"

Gubbins_SNPs_CDSAnno_WiEpitopeOverlap_TSV = f"{Gubbins_V1_OutputDir}/Gubbins.SNPs.AnnoByEvent.AnnoByCDSandEpitope.All.V2.tsv" 


In [197]:
Gub_SNPs_CDSEffect_WiEpiCt_DF = pd.read_csv(Gubbins_SNPs_CDSAnno_WiEpitopeOverlap_TSV, sep ="\t")
Gub_SNPs_CDSEffect_WiEpiCt_DF.shape

(26508, 24)

### c) Parse processed epitope mapping data w/ mutation freq info

In [198]:
Repo_Epitope_MainDir = "../../Data/220813_MtbEpitopes"
LPM_AllAssayedPeptides_WiMutInfo_TSV = f"{Repo_Epitope_MainDir}/240820.Panda24_Lind16.Merged.PeptidesMappedToRv.All.AnnoByMutFreq.V2.tsv" 


In [199]:

LPM_AllPep_V2_DF = pd.read_csv(LPM_AllAssayedPeptides_WiMutInfo_TSV, sep ="\t" )

LPM_PosEpi_V2_DF = LPM_AllPep_V2_DF.query("PosEpitope_Any == True")
LPM_NegPep_V2_DF = LPM_AllPep_V2_DF.query("PosEpitope_Any == False")


In [200]:
LPM_AllPep_V2_DF.shape

(18741, 30)

In [201]:
LPM_PosEpi_V2_DF.shape

(424, 30)

In [202]:
LPM_NegPep_V2_DF.shape

(18317, 30)

### d) Parse gene-level summary stats of GCEs, Mutations, and epitope mapping

In [203]:
Repo_AntigenMut_MainDir = "../../Data/250609.AntigenMutationalBurdenAnalysis"

Rv_Gene_MutStats_V1_TSV = f"{Repo_AntigenMut_MainDir}/250808.RvGenes.GCE.MutationFreq.Epitopes.SummaryStats.V1.tsv"
Rv_Antigens_MutStats_V1_TSV = f"{Repo_AntigenMut_MainDir}/250808.RvL2Antigens.GCE.MutationFreq.Epitopes.SummaryStats.V1.tsv"


In [204]:

RvGenes_GCandMutFreqStats_DF = pd.read_csv(Rv_Gene_MutStats_V1_TSV,
                                           sep = "\t")
RvGenes_GCandMutFreqStats_DF.shape

(3841, 36)

In [205]:

Antigen_GCandMutFreqStats_DF = pd.read_csv(Rv_Antigens_MutStats_V1_TSV,
                                           sep = "\t")
Antigen_GCandMutFreqStats_DF.shape

(53, 38)

### e) Parse Per position Mutation Freq + Epitope mapping coverage DFs

In [206]:
Repo_AntigenMut_MainDir = "../../Data/250609.AntigenMutationalBurdenAnalysis"
PerRvPosition_EpitopeCovStats_TSV = f"{Repo_AntigenMut_MainDir}/250609.PerRvPos.EpitopeCoverage.AssayedPosOnly.V1.tsv.gz"

PerRvCodon_MutationFreq_TSV = f"{Repo_AntigenMut_MainDir}/250609.PerRvCodon.MissenseMutatationFreq.V1.tsv.gz"

In [207]:
Rv_EpitopeCov_NonZero_DF = pd.read_csv(PerRvPosition_EpitopeCovStats_TSV,
                                  sep = "\t")

RvPerCodon_NSMutFreq_DF = pd.read_csv(PerRvCodon_MutationFreq_TSV,
                                  sep = "\t")

# Extra analysis and exploration

## Look at GC event stats w/ info about # of epitopes mutated

In [208]:
pGCE_V3_DF.shape

(324, 51)

In [209]:
pGCE_V3_DF.head(4)

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs,N_LCR_Ovrlap,OvrlapWi_LowComplexityRegion,N_LowPmap_Ovrlap,OvrlapWi_LowPmap,LenOfTaxaList,IsTermNode,Freq_SNP_FoundInAnyPR,NumHmTargets,NumHm_AtMaxKmerMatch,Top_KmerMatch_HomologTarIDs,Top_KmerMatch_HomologGeneIDs,Max_KmerMatch_ToHm,DistToHm_TopKmatch,TotalKmers_Eval,NumHm_AtMaxSNPMatch,Top_SNPMatch_HomologTarIDs,Top_SNPMatch_HomologGeneIDs,Max_SNPMatch_ToHm,DistToHm_TopSNPmatch,MappedEvent,Overlap_WiAntigenLVL2Gene,N_NS_Mut,N_EpiMutated_By_GCE,N_NegPepMutated_By_GCE,EpitopeIDs_NSMutByEvent
0,NC_000962.3,103600,104478,0.0,.,Node_148,Node_133,1737.432787,10,"['mada_2-31', 'mada_1-41', 'MT_0080', 'mada_10...",103599,104038.5,879,NaN,"Rv0093c,Rv0094c","Rv0093c,Rv0094c",False,False,True,False,Event_001,2,1,0,0,PR_HmRegion_002,0,0,1,1,130,False,0.0,2,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.8608,NaN,79,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.0,NaN,True,False,0,0,0,.
1,NC_000962.3,103676,103930,0.0,.,Node_3,N1177,1950.228311,5,['N1177'],103675,103802.5,255,lineage6,Rv0094c,Rv0094c,False,False,True,False,Event_002,2,1,0,0,PR_HmRegion_002,0,0,1,1,1,True,0.0,2,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.0000,NaN,46,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.0,NaN,False,False,4,0,0,.
2,NC_000962.3,103823,103846,0.0,.,Node_146,N0153,1944.165722,4,['N0153'],103822,103834.0,24,lineage1,Rv0094c,Rv0094c,False,False,True,False,Event_003,2,1,0,0,PR_HmRegion_002,0,0,1,1,1,True,0.0,2,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.0000,NaN,28,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.0,NaN,False,False,4,0,0,.
3,NC_000962.3,103823,104065,0.0,.,Node_134,R27252,2067.461166,6,['R27252'],103822,103943.5,243,lineage1,Rv0094c,Rv0094c,False,False,True,False,Event_004,2,1,0,0,PR_HmRegion_002,0,0,1,1,1,True,0.0,2,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.0000,NaN,50,2,"Rv1587c,Rv1588c-H37Rv-1788513-1789865-Rv3466,R...","Rv1587c,Rv1588c-Rv3466,Rv3467",0.0,NaN,False,False,5,0,0,.


In [210]:
pGCE_V3_DF.query("N_EpiMutated_By_GCE > 0 ").shape

(25, 51)

In [211]:
pGCE_V3_DF.query("N_EpiMutated_By_GCE > 0 ").shape

(25, 51)

In [212]:
pGCE_V3_DF.query("N_NegPepMutated_By_GCE > 0 ").shape

(131, 51)

In [213]:
pGCE_V3_DF.query("(Max_KmerMatch_ToHm >= 0.5) & N_EpiMutated_By_GCE > 0 ").shape

(23, 51)

In [214]:
pGCE_V3_DF.query("(Max_KmerMatch_ToHm >= 0.5) & N_EpiMutated_By_GCE > 0 ").shape

(23, 51)

In [215]:
pGCE_V3_DF.query("(Max_KmerMatch_ToHm >= 0.5) & N_NegPepMutated_By_GCE > 0 ").shape

(101, 51)

In [216]:
pGCE_V3_DF[["N_EpiMutated_By_GCE", "N_NegPepMutated_By_GCE"]].describe()

,N_EpiMutated_By_GCE,N_NegPepMutated_By_GCE
count,324.000000,324.000000
mean,0.157407,0.808642
std,0.705950,1.358537
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,1.000000
max,8.000000,12.000000


In [217]:
pGCE_V3_DF.query("Overlap_Genes == 'PPE18'")[["N_EpiMutated_By_GCE", "N_NegPepMutated_By_GCE"]].describe()

,N_EpiMutated_By_GCE,N_NegPepMutated_By_GCE
count,7.000000,7.000000
mean,3.142857,4.714286
std,2.340126,3.817254
min,1.000000,1.000000
25%,2.000000,2.000000
50%,2.000000,4.000000
75%,3.500000,6.000000
max,8.000000,12.000000


In [218]:
pGCE_V3_DF.query("Overlap_Genes == 'PPE18'").head(4)

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs,N_LCR_Ovrlap,OvrlapWi_LowComplexityRegion,N_LowPmap_Ovrlap,OvrlapWi_LowPmap,LenOfTaxaList,IsTermNode,Freq_SNP_FoundInAnyPR,NumHmTargets,NumHm_AtMaxKmerMatch,Top_KmerMatch_HomologTarIDs,Top_KmerMatch_HomologGeneIDs,Max_KmerMatch_ToHm,DistToHm_TopKmatch,TotalKmers_Eval,NumHm_AtMaxSNPMatch,Top_SNPMatch_HomologTarIDs,Top_SNPMatch_HomologGeneIDs,Max_SNPMatch_ToHm,DistToHm_TopSNPmatch,MappedEvent,Overlap_WiAntigenLVL2Gene,N_NS_Mut,N_EpiMutated_By_GCE,N_NegPepMutated_By_GCE,EpitopeIDs_NSMutByEvent
90,NC_000962.3,1339399,1339436,0.0,.,Node_36,Node_28,655.459143,4,"['M0016395_7', 'R15311']",1339398,1339417.0,38,lineage4,PPE18,Rv1196,False,True,False,False,Event_091,2,1,0,0,PR_HmRegion_053,0,0,1,1,2,False,0.75,2,2,"PE31,PPE60-H37Rv-3894016-3895519-PPE19-H37Rv-1...","PE31,PPE60-PPE19",0.5769,NaN,26,1,"PE31,PPE60-H37Rv-3894016-3895519","PE31,PPE60",0.75,1856181.5,True,True,1,2,1,"UnqPeptide_1141-PPE18,UnqPeptide_16789-PPE18"
91,NC_000962.3,1339432,1339516,0.0,.,Node_114,TB3368,475.291405,6,['TB3368'],1339431,1339473.5,85,lineage2,PPE18,Rv1196,False,True,False,False,Event_092,2,1,0,0,PR_HmRegion_053,0,0,1,1,1,True,1.00,2,1,PPE19-H37Rv-1532539-1533632,PPE19,1.0000,193612.0,31,1,"PE31,PPE60-H37Rv-3894016-3895519","PE31,PPE60",0.50,1856238.0,True,True,3,2,4,"UnqPeptide_1141-PPE18,UnqPeptide_16789-PPE18"
92,NC_000962.3,1339432,1339774,0.0,.,Node_27,Node_5,1536.817046,5,"['mada_2-31', 'mada_1-41']",1339431,1339602.5,343,lineage4,PPE18,Rv1196,False,True,False,False,Event_093,2,1,0,0,PR_HmRegion_053,0,0,1,1,2,False,1.00,2,2,"PE31,PPE60-H37Rv-3894016-3895519-PPE19-H37Rv-1...","PE31,PPE60-PPE19",0.7027,NaN,37,1,"PE31,PPE60-H37Rv-3894016-3895519","PE31,PPE60",0.80,1856367.0,True,True,1,2,1,"UnqPeptide_1141-PPE18,UnqPeptide_16789-PPE18"
93,NC_000962.3,1339648,1339806,0.0,.,Node_25,M0017522_5,1625.533450,20,['M0017522_5'],1339647,1339726.5,159,lineage4,PPE18,Rv1196,False,True,False,False,Event_094,2,1,0,0,PR_HmRegion_053,0,0,1,1,1,True,1.00,2,1,"PE31,PPE60-H37Rv-3894016-3895519","PE31,PPE60",1.0000,1856491.0,114,1,"PE31,PPE60-H37Rv-3894016-3895519","PE31,PPE60",1.00,1856491.0,True,True,17,8,6,"UnqPeptide_1812-PPE18,UnqPeptide_5494-PPE18,Un..."


In [219]:
pGCE_V3_DF.query("N_NS_Mut > 0 ").shape

(280, 51)

In [220]:
pGCE_V3_DF.query("N_EpiMutated_By_GCE > 0 ").shape

(25, 51)

In [221]:
pGCE_V3_DF["Overlap_WiAntigenLVL2Gene"].value_counts()

Overlap_WiAntigenLVL2Gene
False    276
True      48
Name: count, dtype: int64

In [222]:
48/213

0.22535211267605634

In [223]:
25/213

0.11737089201877934

In [224]:
LPM_PosEpi_V2_DF[LPM_PosEpi_V2_DF["EpitopeSymbol_ID"].isin(Dict_EventID_To_MutEpiID["Event_094"]) ]   

,Epitope_ID,Epitope_Seq,Epitope_Len,RvID,Symbol,AA_Start,AA_End,Chrom,Rv_Start,Rv_End,EpitopeSeqFreqInAntigen,Dataset,Assayed_Panda24,PosEpitope_Panda24,Assayed_Lindestam16,PosEpitope_Lindestam16,PosEpitope_Any,EpitopeSymbol_ID,N_HmRegion,HasHmRegion,Antigen_LVL2,N_NSMut_Total,N_NSMut_mGCE,IsMutBymGCE,N_NSMut_pGCE,IsMutBypGCE,N_mGCEs_WiNS,WiNS_mGC_EventIDs,N_pGCEs_WiNS,WiNS_pGC_EventIDs
1817,UnqPeptide_1812,LLGQNTPAIAVNEAE,15,Rv1196,PPE18,125,140,NC_000962.3,1339723,1339768,1,Lindestam16_Panda24_Merged,True,False,True,True,True,UnqPeptide_1812-PPE18,1,True,True,4,4,True,4,True,1,Event_094,1,Event_094
5493,UnqPeptide_5494,LIATNLLGQNTPAIA,15,Rv1196,PPE18,120,135,NC_000962.3,1339708,1339753,1,Lindestam16_Panda24_Merged,True,True,True,True,True,UnqPeptide_5494-PPE18,1,True,True,3,3,True,3,True,1,Event_094,1,Event_094
7218,UnqPeptide_7207,TPAIAVNEAEYGEMW,15,Rv1196,PPE18,130,145,NC_000962.3,1339738,1339783,1,Lindestam16_Panda24_Merged,True,True,True,True,True,UnqPeptide_7207-PPE18,1,True,True,6,6,True,6,True,1,Event_094,1,Event_094
8693,UnqPeptide_8677,AELMILIATNLLGQN,15,Rv1196,PPE18,115,130,NC_000962.3,1339693,1339738,1,Lindestam16_Panda24_Merged,True,False,True,True,True,UnqPeptide_8677-PPE18,1,True,True,6,6,True,6,True,1,Event_094,1,Event_094
15833,UnqPeptide_15834,IAENRAELMILIATN,15,Rv1196,PPE18,110,125,NC_000962.3,1339678,1339723,1,Lindestam16_Panda24_Merged,True,False,True,True,True,UnqPeptide_15834-PPE18,1,True,True,6,6,True,6,True,1,Event_094,1,Event_094
16444,UnqPeptide_16444,YGEMWAQDAAAMFGY,15,Rv1196,PPE18,140,155,NC_000962.3,1339768,1339813,1,Lindestam16_Panda24_Merged,True,False,True,True,True,UnqPeptide_16444-PPE18,1,True,True,6,6,True,6,True,1,Event_094,1,Event_094
17227,UnqPeptide_17231,VPPPVIAENRAELMI,15,Rv1196,PPE18,105,120,NC_000962.3,1339663,1339708,1,Lindestam16_Panda24_Merged,True,False,True,True,True,UnqPeptide_17231-PPE18,1,True,True,4,4,True,4,True,1,Event_094,1,Event_094
17385,UnqPeptide_17390,AAYETAYGLTVPPPV,15,Rv1196,PPE18,95,110,NC_000962.3,1339633,1339678,1,Lindestam16_Panda24_Merged,True,False,True,True,True,UnqPeptide_17390-PPE18,1,True,True,3,1,True,1,True,1,Event_094,1,Event_094


## Basic questions to ask of GC events mutating epitopes/antigens

### 1) How many events mutate (cause AA change) in an antigen? (41/213, 19%)

In [225]:
pGCE_V3_WiNSMut_DF = pGCE_V3_DF.query("N_NS_Mut > 0")
pGCE_V3_WiNSMut_DF.shape

(280, 51)

In [226]:
pGCE_V3_DF["Overlap_WiAntigenLVL2Gene"].value_counts()

Overlap_WiAntigenLVL2Gene
False    276
True      48
Name: count, dtype: int64

In [227]:
pGCE_V3_DF.query("N_NS_Mut > 0 & Overlap_WiAntigenLVL2Gene == True").shape

(42, 51)

In [228]:
pGCE_V3_DF.query("N_NS_Mut > 0 & Overlap_WiAntigenLVL2Gene == True")["Overlap_Genes"].value_counts()

Overlap_Genes
PPE60                8
PPE18                7
PPE19                5
esxL                 5
esxP                 3
esxK,esxL            3
PPE46                2
esxM,esxN            2
esxP,Rv2348c         2
PPE18,esxK,esxL      1
esxO,esxP,Rv2348c    1
esxO,esxP            1
esxO                 1
esxN                 1
Name: count, dtype: int64

In [229]:
pGCE_V3_DF.query("N_NS_Mut > 0 & Overlap_WiAntigenLVL2Gene == False")["Overlap_Genes"].value_counts().head(10)

Overlap_Genes
PE_PGRS27    18
PE_PGRS28    14
Rv3466       14
Rv0095c      13
PE_PGRS54    13
Rv1945       11
Rv0094c      11
PE_PGRS4     10
PPE54         8
PE_PGRS18     8
Name: count, dtype: int64

In [230]:
pGCE_V3_DF.query("N_NS_Mut > 0 & Overlap_WiAntigenLVL2Gene == True").shape

(42, 51)

### 2) How many events mutate (AA change) an epitope? (28/213, 13%)

In [231]:
pGCE_V3_DF.query("N_EpiMutated_By_GCE > 0 ").shape

(25, 51)

In [232]:
22/213

0.10328638497652583

In [233]:
27/213

0.1267605633802817

In [234]:
pGCE_V3_DF.query("N_EpiMutated_By_GCE > 0 ")["Overlap_Genes"].value_counts().head(4)

Overlap_Genes
PPE18        7
esxL         5
PPE60        5
esxK,esxL    3
Name: count, dtype: int64

### 3) What are the specific antigens being mutated by GCEs? How many are there?

In [235]:
LPM_PosEpi_V2_DF.head(1)

,Epitope_ID,Epitope_Seq,Epitope_Len,RvID,Symbol,AA_Start,AA_End,Chrom,Rv_Start,Rv_End,EpitopeSeqFreqInAntigen,Dataset,Assayed_Panda24,PosEpitope_Panda24,Assayed_Lindestam16,PosEpitope_Lindestam16,PosEpitope_Any,EpitopeSymbol_ID,N_HmRegion,HasHmRegion,Antigen_LVL2,N_NSMut_Total,N_NSMut_mGCE,IsMutBymGCE,N_NSMut_pGCE,IsMutBypGCE,N_mGCEs_WiNS,WiNS_mGC_EventIDs,N_pGCEs_WiNS,WiNS_pGC_EventIDs
29,UnqPeptide_29,AQAAVVRFQEAANKQ,15,Rv3874,esxB,50,65,NC_000962.3,4352423,4352468,1,Lindestam16_Panda24_Merged,True,False,True,True,True,UnqPeptide_29-esxB,0,False,True,0,0,False,0,False,0,.,0,.
